In [ ]:
# Cell 1: Improved Live Game Data Retrieval

import pandas as pd
import ssl
import requests
from datetime import datetime, timedelta
import traceback
import pytz
from caching.supabase_client import supabase

def get_todays_date() -> str:
    """Return today's date in YYYY-MM-DD format in Pacific Time."""
    pacific_tz = pytz.timezone('America/Los_Angeles')
    return datetime.now(pacific_tz).strftime('%Y-%m-%d')

def get_yesterdays_date() -> str:
    """Return yesterday's date in YYYY-MM-DD format in Pacific Time."""
    pacific_tz = pytz.timezone('America/Los_Angeles')
    return (datetime.now(pacific_tz) - timedelta(days=1)).strftime('%Y-%m-%d')

def fetch_live_games() -> pd.DataFrame:
    """
    Fetch only current live games with proper date filtering.
    First uses today's date; if none found, checks yesterday.
    Filters out games flagged as finished.
    """
    try:
        pacific_tz = pytz.timezone('America/Los_Angeles')
        today = datetime.now(pacific_tz).strftime('%Y-%m-%d')
        yesterday = (datetime.now(pacific_tz) - timedelta(days=1)).strftime('%Y-%m-%d')
        print(f"Fetching live games for {today} (Pacific Time)")
        
        response = supabase.table("nba_live_game_stats").select("*").eq("game_date", today).execute()
        if not response.data:
            print(f"No live games found for today ({today}). Checking yesterday...")
            response = supabase.table("nba_live_game_stats").select("*").eq("game_date", yesterday).execute()
        if not response.data:
            print("No live games found for today or yesterday.")
            return pd.DataFrame()
        
        live_df = pd.DataFrame(response.data)
        # Ensure an 'is_finished' column exists and fill NaNs with False
        if 'is_finished' not in live_df.columns:
            live_df['is_finished'] = False
        else:
            live_df['is_finished'] = live_df['is_finished'].fillna(False)
        # Mark completed games: if home_q4 & away_q4 are nonzero and is_finished True
        completed_mask = (live_df['home_q4'] > 0) & (live_df['away_q4'] > 0) & (live_df['is_finished'])
        active_df = live_df[~completed_mask].copy()
        if active_df.empty:
            print("No active games found – all games appear to be completed.")
            return pd.DataFrame()
        print(f"Found {len(active_df)} active live games")
        return active_df
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def validate_game_data(df: pd.DataFrame) -> pd.DataFrame:
    """Validates and cleans game data to ensure consistency."""
    if df.empty:
        return pd.DataFrame()
    clean_df = df.copy()
    required_cols = ['game_id', 'home_team', 'away_team', 'game_date']
    for col in required_cols:
        if col not in clean_df.columns:
            print(f"Missing required column: {col}")
            return pd.DataFrame()
    # Vectorized team name cleaning
    clean_df['home_team'] = clean_df['home_team'].astype(str).str.strip()
    clean_df['away_team'] = clean_df['away_team'].astype(str).str.strip()
    # Ensure quarter scores are numeric (with fallback 0)
    for q in range(1, 5):
        for prefix in ['home_q', 'away_q']:
            col = f'{prefix}{q}'
            if col in clean_df.columns:
                clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce').fillna(0)
    # If total scores missing, compute them from quarters
    if 'home_score' not in clean_df.columns or clean_df['home_score'].isna().all():
        clean_df['home_score'] = sum(clean_df[f'home_q{q}'] for q in range(1, 5))
    if 'away_score' not in clean_df.columns or clean_df['away_score'].isna().all():
        clean_df['away_score'] = sum(clean_df[f'away_q{q}'] for q in range(1, 5))
    # Determine current quarter (using max quarter with score > 0)
    clean_df['current_quarter'] = 0
    for q in range(1, 5):
        mask = (clean_df[f'home_q{q}'] > 0) | (clean_df[f'away_q{q}'] > 0)
        clean_df.loc[mask, 'current_quarter'] = q
    return clean_df

def normalize_team_name(name: str) -> str:
    """Normalize team names for consistent matching."""
    if not name:
        return ""
    name = str(name).lower().strip()
    mappings = {
        'lakers': 'los angeles lakers',
        'la lakers': 'los angeles lakers',
        'clippers': 'los angeles clippers',
        'la clippers': 'los angeles clippers',
        'blazers': 'portland trail blazers',
        'sixers': 'philadelphia 76ers',
        'philly': 'philadelphia 76ers',
        'knicks': 'new york knicks',
        'ny knicks': 'new york knicks',
        'nets': 'brooklyn nets',
        'mavs': 'dallas mavericks',
        'cavs': 'cleveland cavaliers',
        'wolves': 'minnesota timberwolves',
        't-wolves': 'minnesota timberwolves',
        'celts': 'boston celtics',
        'pels': 'new orleans pelicans'
    }
    for key, value in mappings.items():
        if key in name:
            return value
    return name

def match_teams_to_schedule(live_df: pd.DataFrame, schedule_df: pd.DataFrame) -> pd.DataFrame:
    """Match live game teams to official schedule with normalized team names."""
    if live_df.empty or schedule_df.empty:
        return live_df
    matched_games = []
    for _, game in live_df.iterrows():
        game_dict = game.to_dict()
        home = normalize_team_name(game_dict.get('home_team', ''))
        away = normalize_team_name(game_dict.get('away_team', ''))
        exact = schedule_df[
            ((schedule_df['home_team'].str.lower() == home) & (schedule_df['away_team'].str.lower() == away)) |
            ((schedule_df['home_team'].str.lower() == away) & (schedule_df['away_team'].str.lower() == home))
        ]
        if not exact.empty:
            match = exact.iloc[0]
            game_dict['home_team'] = match['home_team']
            game_dict['away_team'] = match['away_team']
            game_dict['game_id'] = match['game_id']
            game_dict['matched_to_schedule'] = True
        else:
            game_dict['matched_to_schedule'] = False
        matched_games.append(game_dict)
    result_df = pd.DataFrame(matched_games)
    print(f"Matched {result_df['matched_to_schedule'].sum()} of {len(result_df)} games to schedule")
    return result_df

def convert_scheduled_to_live_format(schedule_df: pd.DataFrame) -> pd.DataFrame:
    """Converts scheduled games to the format expected by the prediction system."""
    if schedule_df.empty:
        return pd.DataFrame()
    live_format = []
    for _, game in schedule_df.iterrows():
        live_game = {
            'game_id': game['game_id'],
            'home_team': game['home_team'],
            'away_team': game['away_team'],
            'game_date': game['game_date'],
            'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
            'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
            'home_score': 0, 'away_score': 0,
            'current_quarter': 0,
            'matched_to_schedule': True
        }
        live_format.append(live_game)
    return pd.DataFrame(live_format)

def create_test_games_data() -> pd.DataFrame:
    """Creates synthetic test data when no real games are available."""
    print("Creating synthetic test games data for demonstration")
    today = get_todays_date()
    test_games = [
        {
            'game_id': 90001,
            'home_team': 'Los Angeles Lakers',
            'away_team': 'Brooklyn Nets',
            'game_date': today,
            'home_q1': 30, 'home_q2': 28, 'home_q3': 0, 'home_q4': 0,
            'away_q1': 25, 'away_q2': 32, 'away_q3': 0, 'away_q4': 0,
            'home_score': 58,
            'away_score': 57,
            'current_quarter': 2,
            'matched_to_schedule': False
        },
        {
            'game_id': 90002,
            'home_team': 'Boston Celtics',
            'away_team': 'Miami Heat',
            'game_date': today,
            'home_q1': 28, 'home_q2': 26, 'home_q3': 30, 'home_q4': 0,
            'away_q1': 24, 'away_q2': 27, 'away_q3': 25, 'away_q4': 0,
            'home_score': 84,
            'away_score': 76,
            'current_quarter': 3,
            'matched_to_schedule': False
        }
    ]
    return pd.DataFrame(test_games)

# Test the improved function
live_games = fetch_live_games()
if not live_games.empty:
    print("\nSuccessfully retrieved game data:")
    display(live_games)
else:
    print("\nNo game data available.")


In [ ]:
# Cell 2: Imports and Helper Functions

import pandas as pd
import numpy as np
import joblib
from sqlalchemy import create_engine
import config

def convert_time_to_minutes(time_str: str) -> float:
    """
    Convert a time string "MM:SS" to a float (minutes + seconds/60).
    Returns None if conversion fails.
    """
    if pd.isna(time_str) or ":" not in str(time_str):
        return None
    try:
        minutes, seconds = str(time_str).split(":")
        return float(minutes) + float(seconds) / 60.0
    except Exception as e:
        print(f"Error converting time: {e}")
        return None


In [ ]:
# Cell 2B: Helper functions for data safety and compatibility

def ensure_numeric_features(df, feature_columns):
    """
    Ensures all feature columns are numeric, replacing NaN/None values with appropriate defaults.
    
    Args:
        df (DataFrame): Input DataFrame
        feature_columns (list): List of column names to process
        
    Returns:
        DataFrame: DataFrame with ensured numeric features
    """
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Default values based on column type
    default_map = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'home_score': 0, 'away_score': 0,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'score_ratio': 0.5,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0,
        'cumulative_momentum': 0
    }
    
    # For any column not in default_map, use 0 as default
    for col in feature_columns:
        if col not in default_map:
            default_map[col] = 0
    
    # Process each column
    for col in feature_columns:
        if col in result_df.columns:
            # Convert to numeric, forcing errors to NaN
            result_df[col] = pd.to_numeric(result_df[col], errors='coerce')
            
            # Replace NaN values with appropriate defaults
            result_df[col] = result_df[col].fillna(default_map.get(col, 0))
        else:
            # If column doesn't exist, add it with default values
            result_df[col] = default_map.get(col, 0)
    
    return result_df

def calculate_derived_features(df):
    """
    Calculates derived features like current score from quarters, momentum, etc.
    
    Args:
        df (DataFrame): Input DataFrame with raw game data
        
    Returns:
        DataFrame: DataFrame with additional calculated features
    """
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # 1. Calculate current scores from quarters if not already present
    for idx, row in result_df.iterrows():
        # Home score
        if pd.isna(row.get('home_score')) or row.get('home_score', 0) == 0:
            # Sum up quarters that have data
            home_score = 0
            for q in ['home_q1', 'home_q2', 'home_q3', 'home_q4']:
                if q in row and pd.notna(row[q]):
                    home_score += float(row[q] or 0)
            result_df.at[idx, 'home_score'] = home_score
        
        # Away score
        if pd.isna(row.get('away_score')) or row.get('away_score', 0) == 0:
            # Sum up quarters that have data
            away_score = 0
            for q in ['away_q1', 'away_q2', 'away_q3', 'away_q4']:
                if q in row and pd.notna(row[q]):
                    away_score += float(row[q] or 0)
            result_df.at[idx, 'away_score'] = away_score
    
    # 2. Determine current quarter based on available quarter scores
    for idx, row in result_df.iterrows():
        current_quarter = 0
        if pd.notna(row.get('home_q1')) and float(row.get('home_q1') or 0) > 0:
            current_quarter = 1
        if pd.notna(row.get('home_q2')) and float(row.get('home_q2') or 0) > 0:
            current_quarter = 2
        if pd.notna(row.get('home_q3')) and float(row.get('home_q3') or 0) > 0:
            current_quarter = 3
        if pd.notna(row.get('home_q4')) and float(row.get('home_q4') or 0) > 0:
            current_quarter = 4
        
        # Handle cases where current_quarter might already exist
        if 'current_quarter' not in result_df.columns:
            result_df['current_quarter'] = 0
        result_df.at[idx, 'current_quarter'] = current_quarter
    
    # 3. Calculate score ratio (only for rows with non-zero total score)
    for idx, row in result_df.iterrows():
        home_score = float(row.get('home_score') or 0)
        away_score = float(row.get('away_score') or 0)
        total_score = home_score + away_score
        
        # Default to 0.5 for games with no score yet
        score_ratio = 0.5
        if total_score > 0:
            score_ratio = home_score / total_score
        
        result_df.at[idx, 'score_ratio'] = score_ratio
    
    # 4. Calculate momentum features when applicable
    for idx, row in result_df.iterrows():
        current_quarter = int(row.get('current_quarter') or 0)
        
        # Initialize momentum features with zeros
        result_df.at[idx, 'q1_to_q2_momentum'] = 0
        result_df.at[idx, 'q2_to_q3_momentum'] = 0
        result_df.at[idx, 'q3_to_q4_momentum'] = 0
        result_df.at[idx, 'cumulative_momentum'] = 0
        
        # Calculate quarter-to-quarter momentum shifts
        if current_quarter >= 2:
            # Q1 to Q2 momentum
            home_q1 = float(row.get('home_q1') or 0)
            home_q2 = float(row.get('home_q2') or 0)
            away_q1 = float(row.get('away_q1') or 0)
            away_q2 = float(row.get('away_q2') or 0)
            
            q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
            # Cap extreme values
            q1_to_q2 = max(min(q1_to_q2, 25), -25)
            result_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2
        
        if current_quarter >= 3:
            # Q2 to Q3 momentum
            home_q2 = float(row.get('home_q2') or 0)
            home_q3 = float(row.get('home_q3') or 0)
            away_q2 = float(row.get('away_q2') or 0)
            away_q3 = float(row.get('away_q3') or 0)
            
            q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
            # Cap extreme values
            q2_to_q3 = max(min(q2_to_q3, 25), -25)
            result_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3
        
        if current_quarter >= 4:
            # Q3 to Q4 momentum
            home_q3 = float(row.get('home_q3') or 0)
            home_q4 = float(row.get('home_q4') or 0)
            away_q3 = float(row.get('away_q3') or 0)
            away_q4 = float(row.get('away_q4') or 0)
            
            q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
            # Cap extreme values
            q3_to_q4 = max(min(q3_to_q4, 25), -25)
            result_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4
        
        # Calculate cumulative momentum with weights
        weights = [0.2, 0.3, 0.5]  # Weights for Q1→Q2, Q2→Q3, Q3→Q4
        
        if current_quarter == 2:
            result_df.at[idx, 'cumulative_momentum'] = result_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            q1_to_q2 = result_df.at[idx, 'q1_to_q2_momentum']
            q2_to_q3 = result_df.at[idx, 'q2_to_q3_momentum']
            weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1]) / (weights[0] + weights[1])
            result_df.at[idx, 'cumulative_momentum'] = weighted_momentum
        elif current_quarter >= 4:
            q1_to_q2 = result_df.at[idx, 'q1_to_q2_momentum']
            q2_to_q3 = result_df.at[idx, 'q2_to_q3_momentum']
            q3_to_q4 = result_df.at[idx, 'q3_to_q4_momentum']
            weighted_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1] + q3_to_q4 * weights[2]) / sum(weights)
            result_df.at[idx, 'cumulative_momentum'] = weighted_momentum
        
        # Normalize momentum to [-1, 1]
        if result_df.at[idx, 'cumulative_momentum'] != 0:
            norm_momentum = result_df.at[idx, 'cumulative_momentum'] / 15.0
            result_df.at[idx, 'cumulative_momentum'] = max(min(norm_momentum, 1.0), -1.0)
    
    return result_df

def prepare_features_for_model(df, expected_features, team_avgs=None):
    """
    Prepares features in the correct format for model input.
    
    Args:
        df (DataFrame): Input game data
        expected_features (list): Features expected by the model
        team_avgs (dict, optional): Team scoring averages for rolling features
        
    Returns:
        DataFrame: Features ready for model prediction
    """
    # Ensure all basic numeric features are present and valid
    df = ensure_numeric_features(df, expected_features)
    
    # If team_avgs provided, use them for rolling averages
    if team_avgs is not None and 'rolling_home_score' in expected_features:
        # Update rolling average features if in expected features
        for idx, row in df.iterrows():
            if 'home_team' in row and row['home_team'] in team_avgs:
                df.at[idx, 'rolling_home_score'] = team_avgs[row['home_team']]
            if 'away_team' in row and row['away_team'] in team_avgs:
                df.at[idx, 'rolling_away_score'] = team_avgs[row['away_team']]
    
    # Select only the expected features in the correct order
    X = df[expected_features].copy()
    
    return X

In [ ]:
# Cell 2C: Improved Rest Days Calculation

import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import create_engine
import config
import traceback

def calculate_improved_rest_features(df: pd.DataFrame, max_lookback_days: int = 30, max_rest_days: int = 10) -> pd.DataFrame:
    """
    Calculate rest-related features with better validation and constraints.
    Returns the input DataFrame augmented with rest_days_home, rest_days_away,
    is_back_to_back flags and rest_advantage.
    """
    print(f"Calculating improved rest features with a {max_lookback_days}-day lookback...")
    if df.empty:
        print("No data provided for rest calculation")
        return df
    result_df = df.copy()
    if 'game_date' in result_df.columns:
        result_df['game_date'] = pd.to_datetime(result_df['game_date'], errors='coerce')
        invalid_dates = result_df['game_date'].isna().sum()
        if invalid_dates > 0:
            print(f"Warning: {invalid_dates} invalid dates found; dropping these rows.")
            result_df = result_df.dropna(subset=['game_date'])
    else:
        print("Warning: 'game_date' column not found. Using default rest values.")
        result_df['rest_days_home'] = 2.0
        result_df['rest_days_away'] = 2.0
        result_df['is_back_to_back_home'] = 0
        result_df['is_back_to_back_away'] = 0
        result_df['rest_advantage'] = 0
        return result_df
    
    result_df = result_df.sort_values('game_date')
    teams = set(result_df['home_team'].dropna().unique()).union(set(result_df['away_team'].dropna().unique()))
    print(f"Processing rest days for {len(teams)} teams...")
    team_schedules = {}
    
    for team in teams:
        try:
            if team in team_schedules:
                team_games = team_schedules[team]
            else:
                home_games = result_df[result_df['home_team'] == team][['game_id', 'game_date']].copy()
                home_games['role'] = 'home'
                away_games = result_df[result_df['away_team'] == team][['game_id', 'game_date']].copy()
                away_games['role'] = 'away'
                team_games = pd.concat([home_games, away_games]).sort_values('game_date')
                if team_games.empty:
                    print(f"No games found for team {team}")
                    continue
                team_schedules[team] = team_games
            if len(team_games) <= 1:
                print(f"Only one game found for team {team}; using default rest days.")
                continue
            team_games['prev_date'] = team_games['game_date'].shift(1)
            team_games['days_since_prev'] = (team_games['game_date'] - team_games['prev_date']).dt.days.fillna(3)
            team_games['days_since_prev'] = team_games['days_since_prev'].clip(lower=1, upper=max_rest_days)
            team_games['is_back_to_back'] = (team_games['days_since_prev'] == 1).astype(int)
            for _, row in team_games.iterrows():
                game_id = row['game_id']
                days_rest = row['days_since_prev']
                is_b2b = row['is_back_to_back']
                if row['role'] == 'home':
                    mask = (result_df['game_id'] == game_id) & (result_df['home_team'] == team)
                    result_df.loc[mask, 'rest_days_home'] = days_rest
                    result_df.loc[mask, 'is_back_to_back_home'] = is_b2b
                else:
                    mask = (result_df['game_id'] == game_id) & (result_df['away_team'] == team)
                    result_df.loc[mask, 'rest_days_away'] = days_rest
                    result_df.loc[mask, 'is_back_to_back_away'] = is_b2b
        except Exception as e:
            print(f"Error processing rest days for team {team}: {e}")
            traceback.print_exc()
            continue
    if 'rest_days_home' in result_df.columns and 'rest_days_away' in result_df.columns:
        result_df['rest_advantage'] = (result_df['rest_days_home'] - result_df['rest_days_away']).clip(-5, 5)
    print("Rest day calculation complete.")
    print("\nRest days distribution (home teams):")
    print(result_df['rest_days_home'].value_counts().sort_index())
    print("\nRest days distribution (away teams):")
    print(result_df['rest_days_away'].value_counts().sort_index())
    print("\nBack-to-back games (home teams):", result_df['is_back_to_back_home'].sum())
    print("Back-to-back games (away teams):", result_df['is_back_to_back_away'].sum())
    return result_df

# Test on a sample dataset (for example purposes)
try:
    engine = create_engine(config.DATABASE_URL)
    query = """
    SELECT * FROM nba_historical_game_stats 
    WHERE game_date >= (CURRENT_DATE - INTERVAL '60 days')
    ORDER BY game_date
    LIMIT 100
    """
    sample_df = pd.read_sql(query, engine)
    print(f"Loaded {len(sample_df)} games for rest days calculation test")
    if not sample_df.empty:
        sample_with_rest = calculate_improved_rest_features(sample_df)
        display(sample_with_rest[['rest_days_home', 'rest_days_away', 'rest_advantage',
                                   'is_back_to_back_home', 'is_back_to_back_away']].describe())
    else:
        print("No data found in 'nba_historical_game_stats' for the last 60 days.")
except Exception as e:
    print(f"Error in rest days calculation test: {e}")
    traceback.print_exc()


In [ ]:
# Cell 2D: Robust Model Loading

import os
import joblib
import glob
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
import config
import traceback

def find_model_file(model_name: str = "score_prediction", search_locations: list = None) -> str:
    """Search for a model file by name in common locations."""
    if search_locations is None:
        search_locations = [
            os.getcwd(),
            os.path.join(os.getcwd(), "models"),
            os.path.join(os.getcwd(), "backend", "models"),
            config.MODEL_PATH if hasattr(config, "MODEL_PATH") else None
        ]
        search_locations = [loc for loc in search_locations if loc is not None]
    if hasattr(config, "MODEL_PATH") and os.path.exists(config.MODEL_PATH):
        print(f"Model found at config.MODEL_PATH: {config.MODEL_PATH}")
        return config.MODEL_PATH
    for location in search_locations:
        exacts = [
            os.path.join(location, f"{model_name}.pkl"),
            os.path.join(location, f"{model_name}_model.pkl"),
            os.path.join(location, f"final_{model_name}_model.pkl")
        ]
        for path in exacts:
            if os.path.exists(path):
                print(f"Found exact match: {path}")
                return path
        glob_pattern = os.path.join(location, "*model*.pkl")
        model_files = glob.glob(glob_pattern)
        if model_files:
            newest = max(model_files, key=os.path.getmtime)
            print(f"Found model via glob: {newest}")
            return newest
    print("No model file found in search locations")
    return ""

def create_fallback_model():
    """Creates a simple fallback model for emergency use."""
    print("Training a simple fallback model for emergency use...")
    try:
        model = GradientBoostingRegressor(n_estimators=100, random_state=42)
        np.random.seed(42)
        n_samples = 1000
        X = pd.DataFrame({
            'home_q1': np.random.randint(15, 35, n_samples),
            'home_q2': np.random.randint(15, 35, n_samples),
            'home_q3': np.random.randint(15, 35, n_samples),
            'home_q4': np.random.randint(15, 35, n_samples),
            'score_ratio': np.random.uniform(0.4, 0.6, n_samples),
            'rolling_home_score': np.random.uniform(90, 120, n_samples),
            'rolling_away_score': np.random.uniform(90, 120, n_samples),
            'prev_matchup_diff': np.random.uniform(-10, 10, n_samples)
        })
        y = X['home_q1'] + X['home_q2'] + X['home_q3'] + X['home_q4'] + np.random.normal(0, 5, n_samples)
        model.fit(X, y)
        print("Fallback model trained successfully.")
        return model
    except Exception as e:
        print(f"Error creating fallback model: {e}")
        traceback.print_exc()
        class DummyModel:
            def predict(self, X):
                return np.full(len(X), 105.0)
            def __str__(self):
                return "DUMMY MODEL - Always predicts 105"
        print("WARNING: Using dummy model that always predicts 105")
        return DummyModel()

def load_model_safely(model_path: str = None):
    """Loads a trained model with comprehensive error handling."""
    if model_path is None:
        model_path = find_model_file()
    if model_path and os.path.exists(model_path):
        try:
            model = joblib.load(model_path)
            print(f"Successfully loaded model from: {model_path}")
            if hasattr(model, 'predict') and callable(model.predict):
                if hasattr(model, 'feature_importances_'):
                    fc = len(model.feature_importances_)
                    print(f"Model type: {'enhanced' if fc > 8 else 'original'} ({fc} features)")
                return model
            else:
                print("Loaded object does not have a predict() method.")
        except Exception as e:
            print(f"Error loading model from {model_path}: {e}")
            traceback.print_exc()
    print("Creating fallback model...")
    return create_fallback_model()

def get_feature_list(model=None) -> list:
    """
    Returns the list of features expected by the model.
    Uses the enhanced feature set if the model has more than 8 features.
    """
    original_features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4',
        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
    ]
    enhanced_features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4',
        'score_ratio', 'prev_matchup_diff',
        'rest_days_home', 'rest_days_away', 'rest_advantage',
        'is_back_to_back_home', 'is_back_to_back_away',
        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
    ]
    if model is not None and hasattr(model, 'feature_importances_'):
        if len(model.feature_importances_) > 8:
            return enhanced_features
    return original_features

# Test model loading
model = load_model_safely()
expected_features = get_feature_list(model)
print("\nExpected feature list for this model:")
print(expected_features)


In [ ]:
# Cell 2E: Enhanced Data Consistency Checks

import pandas as pd
import numpy as np
from datetime import datetime

def validate_and_clean_features(df: pd.DataFrame, expected_features: list = None, verbose: bool = True) -> pd.DataFrame:
    """
    Comprehensive validation and cleaning of feature data.
    Ensures required columns exist, converts data types, fills missing values,
    and clips out-of-range values.
    """
    if df.empty:
        if verbose:
            print("No data provided for validation")
        return pd.DataFrame()
    clean_df = df.copy()
    if expected_features is None:
        if 'q1_to_q2_momentum' in clean_df.columns:
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
        else:
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
    if verbose:
        print(f"Validating {len(clean_df)} rows against {len(expected_features)} expected features")
    required_game_cols = ['game_id', 'home_team', 'away_team']
    for col in required_game_cols:
        if col not in clean_df.columns:
            if verbose:
                print(f"Warning: Missing required column: {col}. Adding default values.")
            clean_df[col] = "Unknown" if col != "game_id" else np.arange(1000, 1000+len(clean_df))
    default_values = {
        'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
        'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
        'score_ratio': 0.5,
        'rolling_home_score': 105.0, 'rolling_away_score': 105.0,
        'prev_matchup_diff': 0,
        'rest_days_home': 2, 'rest_days_away': 2, 'rest_advantage': 0,
        'is_back_to_back_home': 0, 'is_back_to_back_away': 0,
        'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0, 'cumulative_momentum': 0,
        'current_quarter': 0,
        'home_score': 0, 'away_score': 0
    }
    for feature in expected_features:
        if feature not in clean_df.columns:
            if verbose:
                print(f"Adding missing feature: {feature} with default {default_values.get(feature, 0)}")
            clean_df[feature] = default_values.get(feature, 0)
        if feature not in ['game_id', 'home_team', 'away_team']:
            clean_df[feature] = pd.to_numeric(clean_df[feature], errors='coerce').fillna(default_values.get(feature, 0))
    # Validate ranges
    validations = [
        ['home_q1', 0, 50, 'quarter score'],
        ['home_q2', 0, 50, 'quarter score'],
        ['home_q3', 0, 50, 'quarter score'],
        ['home_q4', 0, 50, 'quarter score'],
        ['score_ratio', 0, 1, 'score ratio'],
        ['rolling_home_score', 80, 130, 'rolling score'],
        ['rolling_away_score', 80, 130, 'rolling score'],
        ['prev_matchup_diff', -50, 50, 'matchup difference'],
        ['rest_days_home', 1, 10, 'rest days'],
        ['rest_days_away', 1, 10, 'rest days'],
        ['rest_advantage', -5, 5, 'rest advantage'],
        ['q1_to_q2_momentum', -20, 20, 'momentum'],
        ['q2_to_q3_momentum', -20, 20, 'momentum'],
        ['q3_to_q4_momentum', -20, 20, 'momentum'],
        ['cumulative_momentum', -1, 1, 'normalized momentum']
    ]
    for col, min_val, max_val, desc in validations:
        if col in clean_df.columns:
            invalid = ((clean_df[col] < min_val) | (clean_df[col] > max_val)).sum()
            if invalid > 0 and verbose:
                print(f"Clipping {invalid} invalid {desc} values in {col}")
            clean_df[col] = clean_df[col].clip(min_val, max_val)
    # Recalculate current scores and quarter if needed
    if all(f'home_q{i}' in clean_df.columns for i in range(1, 5)):
        clean_df['home_score'] = clean_df[[f'home_q{i}' for i in range(1, 5)]].sum(axis=1)
    if all(f'away_q{i}' in clean_df.columns for i in range(1, 5)):
        clean_df['away_score'] = clean_df[[f'away_q{i}' for i in range(1, 5)]].sum(axis=1)
    if 'current_quarter' not in clean_df.columns:
        clean_df['current_quarter'] = 0
        for idx, row in clean_df.iterrows():
            q_val = 0
            for q in range(1, 5):
                if row.get(f'home_q{q}', 0) > 0 or row.get(f'away_q{q}', 0) > 0:
                    q_val = q
            clean_df.at[idx, 'current_quarter'] = q_val
    if verbose:
        print("\nValidation summary:")
        print(f"- Processed {len(clean_df)} rows")
        feature_stats = pd.DataFrame({
            'min': clean_df[expected_features].min(),
            'max': clean_df[expected_features].max(),
            'mean': clean_df[expected_features].mean(),
            'missing_pct': clean_df[expected_features].isna().mean() * 100
        })
        print("\nFeature statistics after cleaning:")
        display(feature_stats)
    return clean_df

# Test the validation on some problematic data
test_data = pd.DataFrame([
    {
        'game_id': 1001,
        'home_team': 'Boston Celtics',
        'away_team': 'Miami Heat',
        'home_q1': 28, 'home_q2': 30, 'home_q3': None, 'home_q4': 'invalid',
        'away_q1': 26, 'away_q2': 25, 'away_q3': 15, 'away_q4': 18,
        'score_ratio': 2.5,
        'rolling_home_score': 'error',
        'rolling_away_score': 103.5,
        'prev_matchup_diff': -75,
        'rest_days_home': 20,
        'rest_days_away': None
    },
    {
        'game_id': 1002,
        'home_team': 'Los Angeles Lakers',
        'away_team': 'Brooklyn Nets',
        'score_ratio': 0.48,
        'prev_matchup_diff': 5.2
    }
])
cleaned_data = validate_and_clean_features(test_data)
print("\nCleaned test data:")
display(cleaned_data)


In [ ]:
# Cell 3: Get Raw Live Game Data from Supabase with Error Handling

import pandas as pd
import time
from caching.supabase_client import supabase
import traceback

def convert_time_to_minutes(time_str: str) -> float:
    if pd.isna(time_str) or ":" not in str(time_str):
        return None
    try:
        minutes, seconds = str(time_str).split(":")
        return float(minutes) + float(seconds) / 60.0
    except Exception as e:
        print(f"Error converting time: {e}")
        return None

max_retries = 3
retry_delay = 2  # seconds

for attempt in range(max_retries):
    try:
        print(f"Attempting to fetch live game data (attempt {attempt+1}/{max_retries})...")
        response = supabase.table("nba_live_game_stats").select("*").execute()
        raw_data = response.data
        if raw_data:
            raw_df = pd.DataFrame(raw_data)
            if 'minutes' in raw_df.columns:
                raw_df['minutes_numeric'] = raw_df['minutes'].apply(convert_time_to_minutes)
                raw_df.drop(columns=['minutes'], inplace=True)
            print("Latest Raw Game Data:")
            display(raw_df.head())
        else:
            print("No live game data available.")
        break
    except Exception as e:
        print(f"Connection error: {e}")
        if attempt < max_retries - 1:
            print(f"Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
            retry_delay *= 2
        else:
            print("Maximum retries reached. Please check your network connection and Supabase configuration.")
            raw_df = pd.DataFrame()
            break


In [ ]:
# Cell 4: Enhanced Feature Engineering with Improved Rest Day Calculation

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import time
import traceback
import os
from caching.supabase_client import supabase
import config

def normalize_team_name(name: str) -> str:
    """Normalize team names for consistent matching."""
    if not name:
        return ""
    name = str(name).lower().strip()
    mappings = {
        'lakers': 'los angeles lakers',
        'la lakers': 'los angeles lakers',
        'l.a. lakers': 'los angeles lakers',
        'clippers': 'los angeles clippers',
        'la clippers': 'los angeles clippers',
        'blazers': 'portland trail blazers',
        'sixers': 'philadelphia 76ers',
        'philly': 'philadelphia 76ers',
        'knicks': 'new york knicks',
        'nets': 'brooklyn nets'
        # ... (extend as needed)
    }
    for key, value in mappings.items():
        if key in name:
            return value
    return name

def get_improved_rest_data(team: str, game_date: datetime, season_start_dates: dict = None, debug: bool = False) -> dict:
    """
    Enhanced rest day calculation using multiple query strategies.
    Returns a dict with 'rest_days' and 'is_back_to_back'.
    """
    DEFAULT_REST_DAYS = 2
    if isinstance(game_date, str):
        try:
            game_date = pd.to_datetime(game_date)
        except:
            if debug:
                print(f"Warning: Invalid game date format: {game_date}")
            return {'rest_days': DEFAULT_REST_DAYS, 'is_back_to_back': False}
    is_season_start = False
    if season_start_dates:
        game_year = game_date.year
        if game_date.month >= 10:
            season = f"{game_year}-{game_year+1}"
        else:
            season = f"{game_year-1}-{game_year}"
        if season in season_start_dates:
            season_start = pd.to_datetime(season_start_dates[season])
            if (game_date - season_start).days <= 7:
                is_season_start = True
                if debug:
                    print(f"Game is within first week of {season} season")
    lookback_days = 14 if not is_season_start else 180
    lookback_date = (game_date - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    strategies = [
        {"desc": "Original name as home", "query": lambda: 
            supabase.table("nba_historical_game_stats").select("game_date")
            .eq("home_team", team)
            .lt("game_date", game_date.strftime('%Y-%m-%d'))
            .gte("game_date", lookback_date)
            .order('game_date', desc=True)
            .limit(1).execute()
        },
        {"desc": "Original name as away", "query": lambda:
            supabase.table("nba_historical_game_stats").select("game_date")
            .eq("away_team", team)
            .lt("game_date", game_date.strftime('%Y-%m-%d'))
            .gte("game_date", lookback_date)
            .order('game_date', desc=True)
            .limit(1).execute()
        }
    ]
    prev_game_date = None
    for i, strategy in enumerate(strategies):
        try:
            response = strategy["query"]()
            if response.data:
                prev_game_date = pd.to_datetime(response.data[0]['game_date'])
                if debug:
                    print(f"Strategy {i+1} succeeded: Found previous game on {prev_game_date.strftime('%Y-%m-%d')}")
                break
        except Exception as e:
            if debug:
                print(f"Strategy {i+1} ({strategy['desc']}) failed: {e}")
            continue
    if prev_game_date is None:
        if debug:
            print(f"No previous games found for {team} before {game_date.strftime('%Y-%m-%d')}")
        return {'rest_days': 4 if is_season_start else DEFAULT_REST_DAYS, 'is_back_to_back': False}
    rest_days = (game_date - prev_game_date).days
    if rest_days > 10 and not is_season_start:
        if debug:
            print(f"Warning: Unusually high rest days ({rest_days}) for {team}, capping at 10")
        rest_days = 10
    return {'rest_days': rest_days, 'is_back_to_back': rest_days == 1}

def calculate_momentum(home_scores: list, away_scores: list, current_quarter: int) -> dict:
    """
    Calculate momentum based on quarter-to-quarter score changes.
    Returns a dictionary of momentum features.
    """
    momentum = {'q1_to_q2_momentum': 0, 'q2_to_q3_momentum': 0, 'q3_to_q4_momentum': 0, 'cumulative_momentum': 0}
    if current_quarter >= 2 and len(home_scores) >= 2:
        q1_to_q2 = (home_scores[1] - home_scores[0]) - (away_scores[1] - away_scores[0])
        momentum['q1_to_q2_momentum'] = np.clip(q1_to_q2, -15, 15)
    if current_quarter >= 3 and len(home_scores) >= 3:
        q2_to_q3 = (home_scores[2] - home_scores[1]) - (away_scores[2] - away_scores[1])
        momentum['q2_to_q3_momentum'] = np.clip(q2_to_q3, -15, 15)
    if current_quarter >= 4 and len(home_scores) >= 4:
        q3_to_q4 = (home_scores[3] - home_scores[2]) - (away_scores[3] - away_scores[2])
        momentum['q3_to_q4_momentum'] = np.clip(q3_to_q4, -15, 15)
    weights = [0.2, 0.3, 0.5]
    if current_quarter == 2:
        cumulative = momentum['q1_to_q2_momentum']
    elif current_quarter == 3:
        cumulative = (momentum['q1_to_q2_momentum']*weights[0] + momentum['q2_to_q3_momentum']*weights[1]) / (weights[0]+weights[1])
    elif current_quarter >= 4:
        cumulative = (momentum['q1_to_q2_momentum']*weights[0] + momentum['q2_to_q3_momentum']*weights[1] + momentum['q3_to_q4_momentum']*weights[2]) / sum(weights)
    else:
        cumulative = 0
    momentum['cumulative_momentum'] = np.clip(cumulative / 15.0, -1, 1)
    return momentum

def get_previous_matchup_diff(home_team: str, away_team: str, max_lookback: int = 5) -> float:
    """Retrieve the average point differential from previous matchups between two teams."""
    try:
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        matchups = (response_home.data or []) + (response_away.data or [])
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
            diffs = []
            for game in matchups:
                try:
                    if game['home_team'] == home_team and game['away_team'] == away_team:
                        diffs.append(game['home_score'] - game['away_score'])
                    elif game['home_team'] == away_team and game['away_team'] == home_team:
                        diffs.append(game['away_score'] - game['home_score'])
                except Exception as e:
                    print(f"Error processing matchup differential: {e}")
            if diffs:
                return np.clip(np.mean(diffs), -15, 15)
        return 0
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0

def match_live_to_scheduled(live_df: pd.DataFrame, scheduled_df: pd.DataFrame) -> pd.DataFrame:
    """Match live games to official schedule with team name normalization."""
    if live_df.empty or scheduled_df.empty:
        return live_df
    live_df = live_df.copy()
    live_df['verified'] = False
    live_df['home_team_norm'] = live_df['home_team'].astype(str).str.lower().str.strip()
    live_df['away_team_norm'] = live_df['away_team'].astype(str).str.lower().str.strip()
    scheduled_df['home_team_norm'] = scheduled_df['home_team'].astype(str).str.lower().str.strip()
    scheduled_df['away_team_norm'] = scheduled_df['away_team'].astype(str).str.lower().str.strip()
    for idx, game in live_df.iterrows():
        home = game['home_team_norm']
        away = game['away_team_norm']
        match = scheduled_df[
            (scheduled_df['home_team_norm'] == home) &
            (scheduled_df['away_team_norm'] == away)
        ]
        if not match.empty:
            m = match.iloc[0]
            live_df.at[idx, 'game_id'] = m['game_id']
            live_df.at[idx, 'verified'] = True
        else:
            live_df.at[idx, 'verified'] = False
    return live_df

def precompute_enhanced_features(sample_size=None, skip_rest_calc=False, skip_matchup_calc=False):
    """
    Main function to precompute enhanced features for model training.
    
    Args:
        sample_size: Maximum number of games to process (None for all)
        skip_rest_calc: Skip the time-consuming rest day calculations
        skip_matchup_calc: Skip the time-consuming matchup differential calculations
    
    Returns:
        DataFrame with enhanced features
    """
    print("Connecting to database...")
    
    # Define season start dates for rest day calculation
    # Season start dates (approximate - these are the regular season opening days)
    season_start_dates = {
        "2020-2021": "2020-12-22",  # COVID-delayed season
        "2021-2022": "2021-10-19",
        "2022-2023": "2022-10-18",
        "2023-2024": "2023-10-24",
        "2024-2025": "2024-10-22"  # Projected based on typical start dates
    }

    try:
        # Connect to the database
        engine = create_engine(config.DATABASE_URL)
        
        # Load raw historical game data with corrected SQL and optional limit
        if sample_size is not None:
            query = f"SELECT * FROM nba_historical_game_stats ORDER BY game_date LIMIT {sample_size}"
            print(f"Using limited sample of {sample_size} games")
        else:
            query = "SELECT * FROM nba_historical_game_stats ORDER BY game_date"
            
        df = pd.read_sql(query, engine)
        df['game_date'] = pd.to_datetime(df['game_date'])
        print(f"Loaded {len(df)} historical games")
        
        # Convert 'minutes' column to a numeric value if it exists
        if 'minutes' in df.columns:
            def convert_minutes(time_str):
                if pd.isna(time_str) or not isinstance(time_str, str) or ":" not in time_str:
                    return None
                try:
                    minutes, seconds = time_str.split(":")
                    return float(minutes) + float(seconds) / 60.0
                except:
                    return None
            
            df['minutes_numeric'] = df['minutes'].apply(convert_minutes)
            df = df.drop(columns=['minutes'])
        
        # Compute rolling scores for home and away teams
        print("Computing rolling scores...")
        df = df.sort_values(['home_team', 'game_date'])
        df['rolling_home_score'] = df.groupby('home_team')['home_score'].transform(
            lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
        )
        
        df = df.sort_values(['away_team', 'game_date'])
        df['rolling_away_score'] = df.groupby('away_team')['away_score'].transform(
            lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
        )
        
        # Fill NaN values with team or league averages
        home_team_avgs = df.groupby('home_team')['home_score'].mean().to_dict()
        away_team_avgs = df.groupby('away_team')['away_score'].mean().to_dict()
        overall_avg = df['home_score'].mean()
        
        df['rolling_home_score'] = df.apply(
            lambda row: row['rolling_home_score'] if pd.notna(row['rolling_home_score']) 
                      else home_team_avgs.get(row['home_team'], overall_avg),
            axis=1
        )
        
        df['rolling_away_score'] = df.apply(
            lambda row: row['rolling_away_score'] if pd.notna(row['rolling_away_score']) 
                      else away_team_avgs.get(row['away_team'], overall_avg),
            axis=1
        )
        
        # Calculate score ratio
        df['score_ratio'] = df.apply(
            lambda row: row['home_score'] / (row['home_score'] + row['away_score'])
                      if (row['home_score'] + row['away_score']) > 0 else 0.5,
            axis=1
        )
        
        # Handle rest-related features - skip if requested
        if skip_rest_calc:
            print("Skipping time-consuming rest calculations, using default values...")
            df['rest_days_home'] = 2
            df['rest_days_away'] = 2
            df['rest_advantage'] = 0
            df['is_back_to_back_home'] = 0
            df['is_back_to_back_away'] = 0
        else:
            # Calculate improved rest-related features
            print("Calculating rest-related features...")
            rest_features = []
            
            for i, row in df.iterrows():
                home_rest = get_improved_rest_data(
                    row['home_team'], 
                    row['game_date'],
                    season_start_dates=season_start_dates
                )
                
                away_rest = get_improved_rest_data(
                    row['away_team'], 
                    row['game_date'],
                    season_start_dates=season_start_dates
                )
                
                rest_features.append({
                    'rest_days_home': home_rest['rest_days'],
                    'rest_days_away': away_rest['rest_days'],
                    'is_back_to_back_home': int(home_rest['is_back_to_back']),
                    'is_back_to_back_away': int(away_rest['is_back_to_back']),
                    'rest_advantage': home_rest['rest_days'] - away_rest['rest_days']
                })
                
                # Progress update every 500 games
                if (i+1) % 500 == 0:
                    print(f"Processed rest features for {i+1}/{len(df)} games")
            
            # Convert list of dicts to DataFrame and join to main df
            rest_df = pd.DataFrame(rest_features)
            for col in rest_df.columns:
                df[col] = rest_df[col]
            
            print(f"Rest features calculated for {len(df)} games")
        
        # Calculate momentum features
        print("Calculating momentum-related features...")
        for i, row in df.iterrows():
            # For historical data, we know all quarters
            current_quarter = 4
            
            # Extract quarter scores
            home_scores = [row.get(f'home_q{q}', 0) for q in range(1, 5)]
            away_scores = [row.get(f'away_q{q}', 0) for q in range(1, 5)]
            
            # Calculate momentum
            momentum = calculate_momentum(home_scores, away_scores, current_quarter)
            
            # Add to dataframe
            for key, value in momentum.items():
                df.at[i, key] = value
            
            # Progress update every 500 games
            if (i+1) % 500 == 0:
                print(f"Processed momentum features for {i+1}/{len(df)} games")
        
        print(f"Momentum features calculated for {len(df)} games")
        
        # Handle matchup differentials - skip if requested
        if skip_matchup_calc:
            print("Skipping time-consuming matchup calculations, using default values...")
            df['prev_matchup_diff'] = 0
        else:
            # Calculate previous matchup differentials
            print("Calculating previous matchup differences...")
            # Track processed team pairs to avoid redundant calculations
            processed_pairs = set()
            for i, row in df.iterrows():
                home_team = row['home_team']
                away_team = row['away_team']
                game_date = row['game_date']
                
                # Create a consistent team pair key (alphabetical order)
                team_pair = tuple(sorted([home_team, away_team]))
                
                # Skip if this pair was already processed
                if team_pair in processed_pairs:
                    continue
                    
                # Find all games between these teams
                team_games = df[
                    ((df['home_team'] == home_team) & (df['away_team'] == away_team)) |
                    ((df['home_team'] == away_team) & (df['away_team'] == home_team))
                ].sort_values('game_date')
                
                # For each game, calculate the matchup differential based on previous games
                for j, game in team_games.iterrows():
                    game_date = game['game_date']
                    prev_games = team_games[team_games['game_date'] < game_date]
                    
                    if not prev_games.empty:
                        diffs = []
                        for _, prev in prev_games.iterrows():
                            if prev['home_team'] == game['home_team']:
                                # Same home team, direct comparison
                                diff = prev['home_score'] - prev['away_score']
                            else:
                                # Teams reversed, reverse the differential
                                diff = prev['away_score'] - prev['home_score']
                            diffs.append(diff)
                        
                        # Calculate average and cap between -15 and 15
                        avg_diff = sum(diffs) / len(diffs)
                        capped_diff = max(min(avg_diff, 15), -15)
                        
                        # Update the dataframe
                        df.at[j, 'prev_matchup_diff'] = capped_diff
                    else:
                        # No previous matchups
                        df.at[j, 'prev_matchup_diff'] = 0
                
                # Mark this pair as processed
                processed_pairs.add(team_pair)
                
                # Progress update every 50 team pairs
                if len(processed_pairs) % 50 == 0:
                    print(f"Processed matchup differences for {len(processed_pairs)} team pairs")
            
            print(f"Matchup differences calculated for {len(processed_pairs)} team pairs")
            
            # Fill any missing matchup differentials with 0
            df['prev_matchup_diff'] = df['prev_matchup_diff'].fillna(0)
            
            print("Unique prev_matchup_diff values:", df['prev_matchup_diff'].unique())
            print("Number of non-zero values:", (df['prev_matchup_diff'] != 0).sum())
        
        # Feature selection for model training
        feature_cols = [
            'game_id', 'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score',
            'prev_matchup_diff', 'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]
        
        # Extract features DataFrame
        features_df = df[feature_cols].copy()
        
        # Print summary statistics for diagnostic purposes
        print("\nRest features summary:")
        print(features_df[['rest_days_home', 'rest_days_away', 'rest_advantage']].describe())
        
        print("\nMomentum features summary:")
        print(features_df[['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']].describe())
        
        # Show sample of the enhanced features
        print("\nSample of enhanced features:")
        display(features_df.head())
        
        # Save to database if we processed the full dataset
        if sample_size is None and not skip_rest_calc and not skip_matchup_calc:
            features_df.to_sql(
                'nba_enhanced_features', 
                engine, 
                if_exists='replace',
                index=False,
                schema='public'
            )
            print("Features saved to nba_enhanced_features table in database")
        else:
            print("Skipping database save as we're using limited/simplified features")
        
        print("Feature precomputation completed successfully")
        return features_df
        
    except Exception as e:
        print(f"Error during feature precomputation: {e}")
        traceback.print_exc()
        return None

# Replace your function call with this
enhanced_features = precompute_enhanced_features(
    sample_size=500,  # Just process 500 games 
    skip_rest_calc=True,  # Skip the time-consuming rest calculations
    skip_matchup_calc=True  # Skip the time-consuming matchup calculations
)

# Train enhanced model with new features
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

print("Training enhanced in-game prediction model with new features...")
if enhanced_features is not None:
    try:
        X = enhanced_features.drop(columns=['game_id'])
        engine = create_engine(config.DATABASE_URL)
        query = "SELECT game_id, home_score FROM nba_historical_game_stats"
        target_df = pd.read_sql(query, engine)
        merged_df = enhanced_features.merge(target_df, on='game_id')
        print(f"Successfully loaded {len(merged_df)} games from historical database")
        features = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'prev_matchup_diff',
            'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]
        if 'rolling_home_score' in merged_df.columns and 'rolling_away_score' in merged_df.columns:
            features.extend(['rolling_home_score', 'rolling_away_score'])
        target = 'home_score'
        X_train, X_test, y_train, y_test = train_test_split(merged_df[features], merged_df[target], test_size=0.2, random_state=42)
        print(f"Training set: {len(X_train)} samples, Test set: {len(X_test)} samples")
        model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42, subsample=0.8)
        model.fit(X_train, y_train)
        train_mse = mean_squared_error(y_train, model.predict(X_train))
        test_mse = mean_squared_error(y_test, model.predict(X_test))
        r2 = r2_score(y_test, model.predict(X_test))
        print(f"Training MSE: {train_mse:.2f}")
        print(f"Test MSE: {test_mse:.2f}")
        print(f"R² Score: {r2:.4f}")
        enhanced_model_path = 'enhanced_xgb_model.pkl'
        joblib.dump(model, enhanced_model_path)
        print(f"Enhanced model saved to {enhanced_model_path}")
        globals()['model'] = model
        globals()['enhanced_model_trained'] = True
        feature_importance = pd.DataFrame({
            'Feature': features,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False)
        plt.figure(figsize=(12, 8))
        plt.barh(feature_importance['Feature'], feature_importance['Importance'])
        plt.xlabel('Importance')
        plt.ylabel('Feature')
        plt.title('Feature Importance in Enhanced In-Game Model')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error in model training: {e}")
        traceback.print_exc()


In [ ]:
# Cell 5: Updated Model with New Features

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor

MODEL_PATH = 'final_xgb_model.pkl'
try:
    model = joblib.load(config.MODEL_PATH)
    print(f"Model loaded from: {config.MODEL_PATH}")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

original_features = [
    'home_q1', 'home_q2', 'home_q3', 'home_q4',
    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
]
enhanced_features = [
    'home_q1', 'home_q2', 'home_q3', 'home_q4',
    'score_ratio', 'prev_matchup_diff',
    'rest_days_home', 'rest_days_away', 'rest_advantage',
    'is_back_to_back_home', 'is_back_to_back_away',
    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
]
expected_features = enhanced_features if (model is not None and hasattr(model, 'feature_importances_') and len(model.feature_importances_) > 8) else original_features
print(f"Using {'enhanced' if expected_features == enhanced_features else 'original'} feature set for model")
if 'new_features_df' not in globals():
    print("Warning: new_features_df not found! Creating an empty DataFrame.")
    new_features_df = pd.DataFrame()
for feature in expected_features:
    if feature not in new_features_df.columns:
        print(f"Warning: feature '{feature}' not found in new_features_df; adding with default 0.")
        new_features_df[feature] = 0
for feature in expected_features:
    new_features_df[feature] = pd.to_numeric(new_features_df[feature], errors='coerce').fillna(0)
X_features = new_features_df[expected_features].copy()
if model is not None:
    try:
        predictions = model.predict(X_features)
        new_features_df['predicted_home_score'] = predictions
        print("Predictions generated successfully")
        display(new_features_df[['predicted_home_score']].head())
    except Exception as e:
        print(f"Error during prediction: {e}")
        traceback.print_exc()
else:
    print("No model available for predictions")


In [ ]:
# Cell 6: Preprocess data for training with diagnostics
from models.score_prediction import load_training_data, preprocess_data
import pandas as pd
import numpy as np

# Load historical training data
df = load_training_data()
print(f"Historical data loaded. Shape: {df.shape}")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")

# Check for nulls in important columns
null_counts = df[['home_team', 'away_team', 'home_score', 'away_score']].isnull().sum()
print("\nNull counts in key columns:")
print(null_counts)

# Examine data before preprocessing
print("\nSample of raw data:")
display(df.head())

# Preprocess with diagnostic outputs
try:
    # Process the data
    X, y = preprocess_data(df)
    
    # Check shapes and types
    print(f"\nFeatures shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    
    # Examine feature statistics
    print("\nFeature statistics:")
    feature_stats = pd.DataFrame({
        'min': X.min(),
        'max': X.max(),
        'mean': X.mean(),
        'null_count': X.isnull().sum(),
        'zero_count': (X == 0).sum(),
        'zero_percent': (X == 0).sum() / len(X) * 100
    })
    display(feature_stats)
    
    # Special focus on prev_matchup_diff
    if 'prev_matchup_diff' in X.columns:
        print(f"\nprev_matchup_diff analysis:")
        print(f"Non-zero values: {(X['prev_matchup_diff'] != 0).sum()} ({(X['prev_matchup_diff'] != 0).sum() / len(X) * 100:.2f}%)")
        print(f"Unique values: {len(X['prev_matchup_diff'].unique())}")
        print(f"First 10 values: {X['prev_matchup_diff'].head(10).tolist()}")
    
    # Display processed features
    print("\nProcessed features sample:")
    display(X.head())
    
except Exception as e:
    print(f"Error in preprocessing: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 6B: Comprehensive End-to-End Prediction with Detailed Logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import traceback
import sys
import os
import time

# Ensure backend modules are in path
backend_dir = os.path.join(os.getcwd(), "backend")
if os.path.exists(backend_dir) and backend_dir not in sys.path:
    sys.path.append(backend_dir)

try:
    from caching.supabase_client import supabase
    import config
except ImportError as e:
    print(f"Import error: {e}")
    class MockSupabase:
        def table(self, name):
            return self
        def select(self, cols):
            return self
        def eq(self, col, val):
            return self
        def execute(self):
            return type('obj', (object,), {'data': []})
    supabase = MockSupabase()
    config = type('obj', (object,), {'MODEL_PATH': 'models/score_prediction_model.pkl'})

def log_with_timestamp(message: str, level: str = "INFO"):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {level}: {message}")

def handle_exception(e: Exception, context: str = ""):
    log_with_timestamp(f"Error in {context}: {str(e)}", "ERROR")
    traceback.print_exc()
    return None

class NBAGamePredictor:
    """Comprehensive class for NBA game predictions with unified error handling."""
    
    def __init__(self):
        self.model = None
        self.feature_list = None
        self.prediction_history = {}
        log_with_timestamp("Initializing NBA Game Predictor")
    
    def load_model(self):
        try:
            import joblib
            model_paths = [
                config.MODEL_PATH,
                'final_xgb_model.pkl',
                'enhanced_xgb_model.pkl',
                'models/score_prediction_model.pkl',
                'backend/models/score_prediction_model.pkl'
            ]
            for path in model_paths:
                try:
                    if os.path.exists(path):
                        model = joblib.load(path)
                        if hasattr(model, 'predict'):
                            log_with_timestamp(f"Successfully loaded model from {path}")
                            self.model = model
                            if hasattr(model, 'feature_importances_'):
                                fc = len(model.feature_importances_)
                                log_with_timestamp(f"Model type: {'enhanced' if fc > 8 else 'original'} ({fc} features)")
                            return True
                except Exception as e:
                    log_with_timestamp(f"Failed to load model from {path}: {e}", "WARNING")
            log_with_timestamp("No model could be loaded. Creating fallback model.", "WARNING")
            self.create_fallback_model()
            return True
        except Exception as e:
            handle_exception(e, "load_model")
            self.create_fallback_model()
            return False
    
    def create_fallback_model(self):
        try:
            from sklearn.ensemble import GradientBoostingRegressor
            model = GradientBoostingRegressor(n_estimators=100, random_state=42)
            np.random.seed(42)
            n_samples = 1000
            X = pd.DataFrame({
                'home_q1': np.random.randint(20, 30, n_samples),
                'home_q2': np.random.randint(20, 30, n_samples),
                'home_q3': np.random.randint(20, 30, n_samples),
                'home_q4': np.random.randint(20, 30, n_samples),
                'score_ratio': np.random.uniform(0.4, 0.6, n_samples),
                'rolling_home_score': np.random.normal(106, 5, n_samples),
                'rolling_away_score': np.random.normal(103, 5, n_samples),
                'prev_matchup_diff': np.random.normal(3, 8, n_samples)
            })
            y = X['home_q1'] + X['home_q2'] + X['home_q3'] + X['home_q4'] + np.random.normal(0, 5, n_samples)
            model.fit(X, y)
            log_with_timestamp("Fallback model created and trained successfully")
            self.model = model
            return True
        except Exception as e:
            handle_exception(e, "create_fallback_model")
            class DummyModel:
                def predict(self, X):
                    return np.full(len(X), 105.0)
            log_with_timestamp("Created dummy model that always predicts 105", "WARNING")
            self.model = DummyModel()
            return False
    
    def get_feature_list(self) -> list:
        if self.feature_list is not None:
            return self.feature_list
        original_features = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
        ]
        enhanced_features = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'prev_matchup_diff',
            'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]
        if self.model is not None and hasattr(self.model, 'feature_importances_'):
            self.feature_list = enhanced_features if len(self.model.feature_importances_) > 8 else original_features
        else:
            self.feature_list = original_features
        log_with_timestamp(f"Using {'enhanced' if self.feature_list==enhanced_features else 'original'} feature set")
        return self.feature_list
    
    def fetch_scheduled_games(self, date_str: str = None) -> pd.DataFrame:
        try:
            if date_str is None:
                date_str = datetime.now().strftime('%Y-%m-%d')
            response = supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute()
            if response.data:
                log_with_timestamp(f"Found {len(response.data)} scheduled games for {date_str}")
                return pd.DataFrame(response.data)
            else:
                log_with_timestamp(f"No scheduled games found for {date_str}")
                return pd.DataFrame()
        except Exception as e:
            return handle_exception(e, "fetch_scheduled_games")
    
    def fetch_live_games(self) -> pd.DataFrame:
        try:
            today = datetime.now().strftime('%Y-%m-%d')
            yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
            response = supabase.table("nba_live_game_stats").select("*").eq("game_date", today).execute()
            if not response.data:
                response = supabase.table("nba_live_game_stats").select("*").eq("game_date", yesterday).execute()
            if not response.data:
                response = supabase.table("nba_live_game_stats").select("*").limit(5).execute()
            if response.data:
                log_with_timestamp(f"Found {len(response.data)} live games")
                live_df = pd.DataFrame(response.data)
                schedule_df = self.fetch_scheduled_games(today)
                if not schedule_df.empty:
                    live_df = match_teams_to_schedule(live_df, schedule_df)
                return live_df
            else:
                log_with_timestamp("No live games found. Checking scheduled games.", "WARNING")
                schedule_df = self.fetch_scheduled_games(today)
                if not schedule_df.empty:
                    live_format = convert_scheduled_to_live_format(schedule_df)
                    return live_format
                log_with_timestamp("No scheduled games found.", "WARNING")
                return pd.DataFrame()
        except Exception as e:
            return handle_exception(e, "fetch_live_games")
    
    def match_live_to_schedule(self, live_df: pd.DataFrame, schedule_df: pd.DataFrame) -> pd.DataFrame:
        try:
            return match_teams_to_schedule(live_df, schedule_df)
        except Exception as e:
            return handle_exception(e, "match_live_to_schedule")
    
    def get_team_rolling_averages(self, days_lookback: int = 60) -> dict:
        try:
            threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
            if not response.data:
                log_with_timestamp(f"No historical data found within {days_lookback} days", "WARNING")
                return {}
            df = pd.DataFrame(response.data)
            df['game_date'] = pd.to_datetime(df['game_date'])
            df = df.sort_values('game_date')
            team_avgs = {}
            all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
            for team in all_teams:
                home_games = df[df['home_team'] == team][['game_date', 'home_score']].rename(columns={'home_score': 'score'})
                away_games = df[df['away_team'] == team][['game_date', 'away_score']].rename(columns={'away_score': 'score'})
                team_games = pd.concat([home_games, away_games]).sort_values('game_date')
                if not team_games.empty:
                    recent = team_games.tail(5)
                    team_avgs[team] = recent['score'].mean()
                else:
                    team_avgs[team] = 105.0
            return team_avgs
        except Exception as e:
            log_with_timestamp(f"Error in get_team_rolling_averages: {e}", "ERROR")
            return {}
    
    def get_previous_matchup_diff(self, home_team: str, away_team: str, max_lookback: int = 5) -> float:
        try:
            response_home = supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", home_team).eq("away_team", away_team)\
                .order('game_date', desc=True).limit(max_lookback).execute()
            response_away = supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", away_team).eq("away_team", home_team)\
                .order('game_date', desc=True).limit(max_lookback).execute()
            home_matchups = response_home.data or []
            away_matchups = response_away.data or []
            matchups = home_matchups + away_matchups
            if matchups:
                matchups.sort(key=lambda x: x.get('game_date', ''), reverse=True)
                matchups = matchups[:max_lookback]
                diffs = []
                for game in matchups:
                    try:
                        if game['home_team'] == home_team and game['away_team'] == away_team:
                            diffs.append(game['home_score'] - game['away_score'])
                        elif game['home_team'] == away_team and game['away_team'] == home_team:
                            diffs.append(game['away_score'] - game['home_score'])
                    except Exception as ex:
                        log_with_timestamp(f"Error processing matchup: {ex}", "WARNING")
                        continue
                return np.clip(np.mean(diffs) if diffs else 0, -15, 15)
            return 0
        except Exception as e:
            log_with_timestamp(f"Error in get_previous_matchup_diff: {e}", "ERROR")
            return 0
    
    def prepare_features(self, live_games_df: pd.DataFrame) -> pd.DataFrame:
        try:
            if live_games_df.empty:
                log_with_timestamp("No games provided for feature preparation", "WARNING")
                return pd.DataFrame()
            expected = self.get_feature_list()
            team_avgs = self.get_team_rolling_averages()
            features_df = validate_and_clean_features(live_games_df, expected_features=expected, verbose=False)
            return features_df
        except Exception as e:
            return handle_exception(e, "prepare_features")
    
    def predict_game_scores(self, features_df: pd.DataFrame) -> pd.DataFrame:
        try:
            if features_df.empty:
                log_with_timestamp("No features provided for prediction", "WARNING")
                return pd.DataFrame()
            if self.model is None:
                self.load_model()
            expected = self.get_feature_list()
            X = features_df[expected]
            preds = self.model.predict(X)
            results = features_df.copy()
            results['predicted_home_score'] = preds
            team_avgs = self.get_team_rolling_averages()
            for idx, row in results.iterrows():
                diff_weight = min(0.3 + (0.1 * row['current_quarter']), 0.6)
                momentum_adj = row.get('cumulative_momentum', 0) * 3.0
                results.at[idx, 'predicted_away_score'] = max(row['predicted_home_score'] - (row['prev_matchup_diff'] * diff_weight) - momentum_adj, row['away_score'])
                results.at[idx, 'predicted_home_score'] = max(row['predicted_home_score'], row['home_score'])
                score_diff = results.at[idx, 'predicted_home_score'] - results.at[idx, 'predicted_away_score']
                win_prob = 1.0 / (1.0 + np.exp(- (0.05 + 0.15 * min(row['current_quarter'] / 4.0, 1.0)) * score_diff))
                results.at[idx, 'win_probability'] = win_prob
                results.at[idx, 'projected_margin'] = results.at[idx, 'predicted_home_score'] - results.at[idx, 'predicted_away_score']
                results.at[idx, 'total_projected_score'] = results.at[idx, 'predicted_home_score'] + results.at[idx, 'predicted_away_score']
                if row['current_quarter'] <= 4:
                    results.at[idx, 'time_remaining'] = 12 * (4 - row['current_quarter'])
                else:
                    results.at[idx, 'time_remaining'] = 0
            return results
        except Exception as e:
            return handle_exception(e, "predict_game_scores")
    
    def visualize_predictions(self, predictions_df: pd.DataFrame):
        try:
            if predictions_df.empty:
                log_with_timestamp("No predictions to visualize", "WARNING")
                return
            n_games = len(predictions_df)
            fig, axs = plt.subplots(n_games, 2, figsize=(16, 5 * n_games))
            if n_games == 1:
                axs = np.array([axs])
            for i, (_, game) in enumerate(predictions_df.iterrows()):
                ax_scores = axs[i, 0]
                teams = [game['home_team'], game['away_team']]
                current_scores = [game.get('current_home_score', 0), game.get('current_away_score', 0)]
                predicted_scores = [game['predicted_home_score'], game['predicted_away_score']]
                x = np.arange(len(teams))
                width = 0.35
                ax_scores.bar(x - width/2, current_scores, width, label='Current')
                ax_scores.bar(x + width/2, predicted_scores, width, label='Predicted Final')
                ax_scores.set_xticks(x)
                ax_scores.set_xticklabels(teams)
                ax_scores.legend()
                confidence = game.get('prediction_confidence', 50)
                quarter_text = "Pre-game" if game['current_quarter'] == 0 else f"Q{game['current_quarter']}"
                ax_scores.set_title(f"{game['home_team']} vs {game['away_team']} - {quarter_text} (Confidence: {confidence}%)")
                for j, v in enumerate(current_scores):
                    ax_scores.text(j - width/2, v + 1, str(int(v)), ha='center')
                for j, v in enumerate(predicted_scores):
                    ax_scores.text(j + width/2, v + 1, f"{v:.1f}", ha='center')
                ax_history = axs[i, 1]
                game_id = game['game_id']
                if game_id in self.prediction_history and len(self.prediction_history[game_id]) > 1:
                    history = pd.DataFrame(self.prediction_history[game_id])
                    ax_history.plot(history['timestamp'], history['predicted_home_score'], label=f"{game['home_team']} Final", marker='o')
                    ax_history.plot(history['timestamp'], history['predicted_away_score'], label=f"{game['away_team']} Final", marker='s')
                    ax_history.set_title("Prediction Evolution")
                    ax_history.set_xlabel("Time")
                    ax_history.set_ylabel("Predicted Score")
                    ax_history.legend()
                    ax_history.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%H:%M'))
                    plt.setp(ax_history.xaxis.get_majorticklabels(), rotation=45)
                else:
                    ax_history.text(0.5, 0.5, "Not enough prediction history yet", ha='center', va='center', transform=ax_history.transAxes)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
            print(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("=" * 80)
            for _, game in predictions_df.iterrows():
                quarter_text = "Pre-game" if game['current_quarter'] == 0 else f"Quarter {game['current_quarter']}"
                print(f"\n{game['home_team']} vs {game['away_team']} - {quarter_text}")
                if game['current_quarter'] > 0:
                    print(f"Current Score: {game['home_team']} {game.get('current_home_score', game.get('home_score', 0)):.0f} - {game['away_team']} {game.get('current_away_score', game.get('away_score', 0)):.0f}")
                print(f"Predicted Final: {game['home_team']} {game['predicted_home_score']:.1f} - {game['away_team']} {game['predicted_away_score']:.1f}")
                if 'win_probability' in game:
                    print(f"Win Probability: {game['win_probability']*100:.1f}%")
                print("-" * 80)
        except Exception as e:
            handle_exception(e, "visualize_predictions")
    
    def generate_recommendations(self, predictions_df: pd.DataFrame) -> dict:
        try:
            if predictions_df.empty:
                log_with_timestamp("No predictions for recommendations", "WARNING")
                return {}
            recommendations = {}
            for _, game in predictions_df.iterrows():
                game_key = f"{game['home_team']} vs {game['away_team']}"
                win_prob = float(game.get('win_probability', 0.5))
                momentum = float(game.get('cumulative_momentum', 0))
                projected_margin = float(game['predicted_home_score'] - game['predicted_away_score'])
                total_projected = float(game['predicted_home_score'] + game['predicted_away_score'])
                quarter = int(game.get('current_quarter', 0))
                recs = {}
                if quarter >= 3 and win_prob > 0.9:
                    recs["betting_tip"] = f"Strong confidence in {game['home_team']} win ({win_prob*100:.1f}%)"
                elif win_prob > 0.75:
                    recs["betting_tip"] = f"Home advantage favors {game['home_team']} ({win_prob*100:.1f}%)"
                elif win_prob < 0.3:
                    recs["betting_tip"] = f"Consider {game['away_team']} for the upset ({(1-win_prob)*100:.1f}%)"
                else:
                    recs["betting_tip"] = "Competitive game; consider hedging."
                if momentum > 0.3:
                    recs["momentum_advice"] = f"Strong momentum for {game['home_team']}"
                elif momentum < -0.3:
                    recs["momentum_advice"] = f"Strong momentum for {game['away_team']}"
                else:
                    recs["momentum_advice"] = "Momentum appears balanced."
                if abs(projected_margin) >= 8:
                    fav = game['home_team'] if projected_margin > 0 else game['away_team']
                    recs["spread_tip"] = f"{fav} projected to cover by {abs(projected_margin):.1f} points."
                else:
                    recs["spread_tip"] = f"Tight margin projected ({abs(projected_margin):.1f} points)."
                league_avg_total = 220
                if total_projected > league_avg_total + 10:
                    recs["over_under_tip"] = f"High-scoring game likely ({total_projected:.1f} points)."
                elif total_projected < league_avg_total - 10:
                    recs["over_under_tip"] = f"Low-scoring game likely ({total_projected:.1f} points)."
                else:
                    recs["over_under_tip"] = f"Projected total: {total_projected:.1f} points."
                if quarter == 4 and (4 - quarter)*12 < 5 and abs(projected_margin) < 6:
                    recs["clutch_tip"] = "Close game in final minutes – monitor closely."
                if quarter < 2:
                    recs["early_game_tip"] = "Early game – watch for adjustments."
                recommendations[game_key] = recs
            return recommendations
        except Exception as e:
            handle_exception(e, "generate_recommendations")
            return {}
    
    def run_full_prediction_pipeline(self) -> pd.DataFrame:
        try:
            if self.model is None:
                if not self.load_model():
                    log_with_timestamp("Failed to load model", "ERROR")
                    return pd.DataFrame()
            live_games_df = self.fetch_live_games()
            if live_games_df.empty:
                log_with_timestamp("No games available for prediction", "WARNING")
                return pd.DataFrame()
            features_df = self.prepare_features(live_games_df)
            if features_df.empty:
                log_with_timestamp("Failed to prepare features", "ERROR")
                return pd.DataFrame()
            predictions_df = self.predict_game_scores(features_df)
            if predictions_df.empty:
                log_with_timestamp("Failed to generate predictions", "ERROR")
                return pd.DataFrame()
            self.visualize_predictions(predictions_df)
            recs = self.generate_recommendations(predictions_df)
            print("\nGame Recommendations:")
            print("=" * 80)
            for game_name, rec in recs.items():
                print(f"\n{game_name}:")
                for key, value in rec.items():
                    print(f"  • {key}: {value}")
                print("-" * 40)
            return predictions_df
        except Exception as e:
            handle_exception(e, "run_full_prediction_pipeline")
            return pd.DataFrame()

# For testing, instantiate and run the predictor (do not run the loop in interactive mode)
predictor = NBAGamePredictor()
preds = predictor.run_full_prediction_pipeline()


In [ ]:
# Cell 7: Enhanced Live Prediction Function with Momentum and Win Probability

from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import joblib
import math
import traceback

def calculate_win_probability(home_score: float, away_score: float, quarter: int, time_remaining: float = None) -> float:
    """
    Calculate win probability for the home team using a logistic function.
    """
    score_diff = home_score - away_score
    if time_remaining is not None:
        total_time = 48.0
        elapsed = total_time - time_remaining
        progress = elapsed / total_time
    else:
        progress = min(quarter / 4.0, 1.0)
    k = 0.05 + (progress * 0.15)
    return 1.0 / (1.0 + math.exp(-k * score_diff))

def calculate_momentum(home_q1, home_q2, home_q3, home_q4, away_q1, away_q2, away_q3, away_q4, current_quarter: int) -> float:
    """Calculate normalized cumulative momentum."""
    h = [float(x or 0) for x in [home_q1, home_q2, home_q3, home_q4]]
    a = [float(x or 0) for x in [away_q1, away_q2, away_q3, away_q4]]
    momentum = 0
    if current_quarter >= 2:
        m1 = (h[1] - h[0]) - (a[1] - a[0])
    else:
        m1 = 0
    if current_quarter >= 3:
        m2 = (h[2] - h[1]) - (a[2] - a[1])
    else:
        m2 = 0
    if current_quarter >= 4:
        m3 = (h[3] - h[2]) - (a[3] - a[2])
    else:
        m3 = 0
    if current_quarter == 2:
        momentum = m1
    elif current_quarter == 3:
        momentum = (m1*0.2 + m2*0.3) / (0.2+0.3)
    elif current_quarter >= 4:
        momentum = (m1*0.2 + m2*0.3 + m3*0.5) / (0.2+0.3+0.5)
    return np.clip(momentum/15.0, -1, 1)

# Example usage:
home_prob = calculate_win_probability(110, 105, 3)
momentum_val = calculate_momentum(28, 27, 22, 0, 24, 29, 26, 0, 3)
print(f"Win Probability: {home_prob:.2f}, Normalized Momentum: {momentum_val:.2f}")


In [ ]:
# Cell 7B: Comprehensive Monitoring System with Automatic Scheduling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta
from IPython.display import clear_output
import traceback
import threading
import json
import os

class NBAGameMonitor:
    """
    System for monitoring NBA games with automated updates and prediction tracking.
    Builds on the NBAGamePredictor to provide continuous monitoring.
    """
    
    def __init__(self, update_interval=60, auto_save=True, debug=True):
        """
        Initialize the monitor with configurable settings.
        
        Args:
            update_interval: Seconds between updates (default: 60)
            auto_save: Whether to auto-save prediction history (default: True)
            debug: Enable detailed debug logging (default: True)
        """
        self.predictor = NBAGamePredictor()  # Assuming NBAGamePredictor is defined elsewhere
        self.update_interval = update_interval
        self.auto_save = auto_save
        self.debug = debug
        self.running = False
        self.update_thread = None
        self.prediction_history = {}
        self.update_count = 0
        self.last_update_time = None
        self.validation_results = None
        
        # File paths for saving data
        self.history_file = "prediction_history.json"
        self.validation_file = "model_validation.json"
        
        # Create directories if needed
        os.makedirs("data", exist_ok=True)
        
        # Load previous history if it exists
        self.load_history()
    
    def log(self, message, level="INFO"):
        """Log message with timestamp."""
        if not self.debug and level == "DEBUG":
            return
            
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {level}: {message}")
    
    def load_history(self):
        """Load prediction history from file if available."""
        history_path = os.path.join("data", self.history_file)
        
        if os.path.exists(history_path):
            try:
                with open(history_path, 'r') as f:
                    raw_history = json.load(f)
                
                # Convert string timestamps back to datetime objects
                for game_id, predictions in raw_history.items():
                    for pred in predictions:
                        if 'timestamp' in pred and isinstance(pred['timestamp'], str):
                            pred['timestamp'] = datetime.fromisoformat(pred['timestamp'])
                
                self.prediction_history = raw_history
                self.log(f"Loaded prediction history for {len(self.prediction_history)} games")
            except Exception as e:
                self.log(f"Error loading prediction history: {e}", "ERROR")
    
    def save_history(self):
        """Save prediction history to file."""
        if not self.prediction_history:
            return
            
        history_path = os.path.join("data", self.history_file)
        
        try:
            # Convert datetime objects to ISO format strings for JSON serialization
            serializable_history = {}
            
            for game_id, predictions in self.prediction_history.items():
                serializable_predictions = []
                for pred in predictions:
                    pred_copy = pred.copy()
                    if 'timestamp' in pred_copy and isinstance(pred_copy['timestamp'], datetime):
                        pred_copy['timestamp'] = pred_copy['timestamp'].isoformat()
                    serializable_predictions.append(pred_copy)
                
                serializable_history[str(game_id)] = serializable_predictions
            
            with open(history_path, 'w') as f:
                json.dump(serializable_history, f)
            
            self.log(f"Saved prediction history to {history_path}")
        except Exception as e:
            self.log(f"Error saving prediction history: {e}", "ERROR")
    
    def validate_model(self, num_test_games=20):
        """Run validation on historical games to assess model performance."""
        self.log("Running model validation...")
        
        try:
            # Validate that the model is loaded
            if self.predictor.model is None:
                self.predictor.load_model()
            
            # Fetch historical games
            current_date = datetime.now().strftime('%Y-%m-%d')
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .lt("game_date", current_date)\
                .order('game_date', desc=True)\
                .limit(num_test_games).execute()
            
            if not response.data:
                self.log("No historical games found for validation", "WARNING")
                return
            
            # Process historical games
            historical_games = response.data
            validation_results = []
            
            self.log(f"Validating model on {len(historical_games)} historical games")
            
            for game in historical_games:
                # Get actual final scores
                actual_home_score = game['home_score']
                actual_away_score = game['away_score']
                game_id = game['game_id']
                
                # Test prediction from each quarter
                quarter_results = []
                
                for test_quarter in range(1, 5):
                    # Create simulated in-progress game
                    sim_game = {
                        'game_id': game_id,
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': test_quarter
                    }
                    
                    # Add quarter scores up to test_quarter
                    for q in range(1, 5):
                        q_key_home = f'home_q{q}'
                        q_key_away = f'away_q{q}'
                        
                        if q <= test_quarter:
                            sim_game[q_key_home] = game.get(q_key_home, 0)
                            sim_game[q_key_away] = game.get(q_key_away, 0)
                        else:
                            sim_game[q_key_home] = 0
                            sim_game[q_key_away] = 0
                    
                    # Calculate current score
                    sim_game['home_score'] = sum([sim_game.get(f'home_q{q}', 0) or 0 for q in range(1, test_quarter+1)])
                    sim_game['away_score'] = sum([sim_game.get(f'away_q{q}', 0) or 0 for q in range(1, test_quarter+1)])
                    
                    # Make prediction
                    features_df = self.predictor.prepare_features(pd.DataFrame([sim_game]))
                    if features_df.empty:
                        continue
                        
                    prediction_result = self.predictor.predict_game_scores(features_df)
                    if prediction_result.empty:
                        continue
                    
                    pred_row = prediction_result.iloc[0]
                    predicted_home = pred_row['predicted_home_final']
                    predicted_away = pred_row['predicted_away_final']
                    
                    # Calculate errors
                    home_error = predicted_home - actual_home_score
                    away_error = predicted_away - actual_away_score
                    
                    quarter_results.append({
                        'quarter': test_quarter,
                        'actual_home': actual_home_score,
                        'actual_away': actual_away_score,
                        'predicted_home': predicted_home,
                        'predicted_away': predicted_away,
                        'home_error': home_error,
                        'away_error': away_error,
                        'absolute_error': (abs(home_error) + abs(away_error)) / 2
                    })
                
                validation_results.append({
                    'game_id': game_id,
                    'home_team': game['home_team'],
                    'away_team': game['away_team'],
                    'quarter_results': quarter_results
                })
            
            # Save validation results
            self.validation_results = validation_results
            
            # Calculate and display summary metrics
            self.display_validation_results()
            
            # Save to file
            validation_path = os.path.join("data", self.validation_file)
            with open(validation_path, 'w') as f:
                json.dump(validation_results, f)
            
            self.log(f"Validation results saved to {validation_path}")
            
            return validation_results
            
        except Exception as e:
            self.log(f"Error during model validation: {e}", "ERROR")
            traceback.print_exc()
            return None
    
    def display_validation_results(self):
        """Display and visualize validation results."""
        if not self.validation_results:
            self.log("No validation results to display", "WARNING")
            return
        
        # Extract metrics by quarter
        quarter_errors = {1: [], 2: [], 3: [], 4: []}
        
        for game in self.validation_results:
            for qr in game['quarter_results']:
                quarter = qr['quarter']
                quarter_errors[quarter].append(qr['absolute_error'])
        
        # Calculate average errors by quarter
        avg_errors = {}
        for quarter, errors in quarter_errors.items():
            if errors:
                avg_errors[quarter] = sum(errors) / len(errors)
            else:
                avg_errors[quarter] = None
        
        # Display results
        print("\nModel Validation Results:")
        print("=" * 60)
        print("Average Prediction Error by Quarter:")
        for quarter, avg_error in avg_errors.items():
            if avg_error is not None:
                print(f"  Quarter {quarter}: {avg_error:.2f} points")
        
        # Visualization
        plt.figure(figsize=(10, 6))
        quarters = list(avg_errors.keys())
        errors = [avg_errors[q] for q in quarters if avg_errors[q] is not None]
        
        bars = plt.bar(quarters, errors, color='salmon')
        plt.xlabel('Quarter')
        plt.ylabel('Average Absolute Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(quarters)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.2f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    
    def start_monitoring(self, max_updates=None, run_validation=True):
        """Start the monitoring process."""
        self.running = True
        self.update_count = 0
        
        # Run model validation if requested
        if run_validation:
            self.validate_model()
        
        # Ensure the model is loaded
        if self.predictor.model is None:
            self.predictor.load_model()
        
        self.log(f"Starting monitoring with {self.update_interval} second updates")
        
        # Begin update loop
        while self.running:
            try:
                self.update_count += 1
                self.last_update_time = datetime.now()
                
                self.log(f"Update #{self.update_count} at {self.last_update_time.strftime('%H:%M:%S')}")
                
                # Clear previous output
                clear_output(wait=True)
                
                # Run full prediction pipeline
                prediction_results = self.predictor.run_full_prediction_pipeline()
                
                # Update prediction history
                if prediction_results is not None and not prediction_results.empty:
                    for _, pred in prediction_results.iterrows():
                        game_id = pred['game_id']
                        if game_id not in self.prediction_history:
                            self.prediction_history[game_id] = []
                        
                        # Add timestamp if missing
                        if 'timestamp' not in pred:
                            pred['timestamp'] = self.last_update_time
                        
                        # Store prediction
                        self.prediction_history[game_id].append(pred.to_dict())
                
                # Save history if auto-save is enabled
                if self.auto_save:
                    self.save_history()
                
                # Check if we've reached max updates
                if max_updates and self.update_count >= max_updates:
                    self.log(f"Reached maximum updates ({max_updates})")
                    break
                
                # Wait for next update
                if self.running and (max_updates is None or self.update_count < max_updates):
                    self.log(f"Waiting {self.update_interval} seconds until next update...")
                    time.sleep(self.update_interval)
                
            except KeyboardInterrupt:
                self.log("Monitoring stopped by user")
                self.running = False
                break
            except Exception as e:
                self.log(f"Error during monitoring update: {e}", "ERROR")
                traceback.print_exc()
                
                # Don't stop on errors, just wait for next update
                time.sleep(self.update_interval)
        
        self.running = False
        self.log("Monitoring stopped")
    
    def start_async_monitoring(self, max_updates=None, run_validation=True):
        """Start monitoring in a background thread."""
        if self.update_thread is not None and self.update_thread.is_alive():
            self.log("Monitoring already running")
            return
        
        self.update_thread = threading.Thread(
            target=self.start_monitoring,
            args=(max_updates, run_validation)
        )
        self.update_thread.daemon = True
        self.update_thread.start()
        self.log("Monitoring started in background thread")
    
    def stop_monitoring(self):
        """Stop the monitoring process."""
        self.running = False
        self.log("Stopping monitoring...")
        
        if self.update_thread is not None and self.update_thread.is_alive():
            self.update_thread.join(timeout=5)
            if self.update_thread.is_alive():
                self.log("Monitoring thread is still running", "WARNING")
        
        self.save_history()
        self.log("Monitoring stopped and history saved")
    
    def get_prediction_accuracy(self):
        """Calculate prediction accuracy for completed games."""
        if not self.prediction_history:
            self.log("No prediction history available")
            return None
        
        accuracy_results = []
        
        for game_id, predictions in self.prediction_history.items():
            # Check if this is a completed game with actual results
            if predictions and 'actual_home_final' in predictions[-1]:
                final_pred = predictions[-1]
                actual_home = final_pred['actual_home_final']
                actual_away = final_pred['actual_away_final']
                
                # Calculate accuracy for each prediction in the game
                game_accuracy = []
                
                for i, pred in enumerate(predictions):
                    predicted_home = pred['predicted_home_final']
                    predicted_away = pred['predicted_away_final']
                    quarter = pred['current_quarter']
                    
                    # Calculate errors
                    home_error = abs(predicted_home - actual_home)
                    away_error = abs(predicted_away - actual_away)
                    avg_error = (home_error + away_error) / 2
                    
                    # Check if prediction got winner correct
                    actual_winner = 'home' if actual_home > actual_away else 'away'
                    predicted_winner = 'home' if predicted_home > predicted_away else 'away'
                    winner_correct = (actual_winner == predicted_winner)
                    
                    game_accuracy.append({
                        'prediction_number': i + 1,
                        'quarter': quarter,
                        'home_error': home_error,
                        'away_error': away_error,
                        'avg_error': avg_error,
                        'winner_correct': winner_correct
                    })
                
                accuracy_results.append({
                    'game_id': game_id,
                    'home_team': predictions[0]['home_team'],
                    'away_team': predictions[0]['away_team'],
                    'predictions': game_accuracy
                })
        
        # Create summary
        if accuracy_results:
            self.display_accuracy_results(accuracy_results)
        
        return accuracy_results
    
    def display_accuracy_results(self, accuracy_results):
        """Display accuracy results."""
        if not accuracy_results:
            return
        
        print("\nPrediction Accuracy Analysis")
        print("=" * 60)
        
        # Overall stats
        total_predictions = 0
        correct_winner_count = 0
        error_by_quarter = {0: [], 1: [], 2: [], 3: [], 4: []}
        
        for game in accuracy_results:
            for pred in game['predictions']:
                total_predictions += 1
                if pred['winner_correct']:
                    correct_winner_count += 1
                
                quarter = pred['quarter']
                error_by_quarter[quarter].append(pred['avg_error'])
        
        # Calculate winner accuracy
        winner_accuracy = (correct_winner_count / total_predictions) if total_predictions > 0 else 0
        print(f"Overall Winner Prediction Accuracy: {winner_accuracy:.1%}")
        
        # Calculate average error by quarter
        print("\nAverage Error by Quarter:")
        for quarter, errors in error_by_quarter.items():
            if errors:
                avg_error = sum(errors) / len(errors)
                print(f"  Quarter {quarter}: {avg_error:.2f} points")
        
        # Visualize
        plt.figure(figsize=(12, 8))
        
        # First subplot - error by quarter
        plt.subplot(2, 1, 1)
        quarters = []
        avg_errors = []
        
        for quarter, errors in error_by_quarter.items():
            if errors:
                quarters.append(quarter)
                avg_errors.append(sum(errors) / len(errors))
        
        bars = plt.bar(quarters, avg_errors, color='lightblue')
        plt.xlabel('Quarter')
        plt.ylabel('Average Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(quarters)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.2f}', ha='center', va='bottom')
        
        # Second subplot - winner accuracy by quarter
        plt.subplot(2, 1, 2)
        quarters = []
        accuracies = []
        
        for quarter in range(5):
            correct = 0
            total = 0
            
            for game in accuracy_results:
                for pred in game['predictions']:
                    if pred['quarter'] == quarter:
                        total += 1
                        if pred['winner_correct']:
                            correct += 1
            
            if total > 0:
                quarters.append(quarter)
                accuracies.append(correct / total)
        
        bars = plt.bar(quarters, accuracies, color='lightgreen')
        plt.xlabel('Quarter')
        plt.ylabel('Accuracy')
        plt.title('Winner Prediction Accuracy by Quarter')
        plt.xticks(quarters)
        plt.ylim(0, 1)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.2%}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

def match_games_to_schedule(live_games_df, schedule_df):
    """Improved function to match games to schedule with better validation"""
    if live_games_df.empty or schedule_df.empty:
        return live_games_df
    
    # Make a copy to avoid modifying the original
    matched_games = []
    
    # Process the schedule first
    for _, scheduled_game in schedule_df.iterrows():
        schedule_game_id = scheduled_game['game_id']
        schedule_home = scheduled_game['home_team']
        schedule_away = scheduled_game['away_team']
        schedule_date = scheduled_game['game_date']
        
        # Try to find this scheduled game in live data
        matching_live = live_games_df[
            (live_games_df['home_team'] == schedule_home) & 
            (live_games_df['away_team'] == schedule_away)
        ]
        
        if not matching_live.empty:
            # Found match - use the scheduled game ID
            live_game = matching_live.iloc[0].to_dict()
            live_game['game_id'] = schedule_game_id
            live_game['verified'] = True
            matched_games.append(live_game)
        else:
            # No live data for this scheduled game - create template
            template = {
                'game_id': schedule_game_id,
                'home_team': schedule_home,
                'away_team': schedule_away,
                'game_date': schedule_date,
                'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
                'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
                'home_score': 0, 'away_score': 0,
                'current_quarter': 0,
                'verified': True,
                'game_status': 'SCHEDULED'
            }
            matched_games.append(template)
    
    return pd.DataFrame(matched_games)

# For testing, we'll need to define a basic NBAGamePredictor class
class NBAGamePredictor:
    def __init__(self):
        self.model = None
        print("[{}] INFO: Initializing NBA Game Predictor".format(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
        
    def load_model(self):
        # Placeholder for model loading
        self.model = "mock_model"
        
    def prepare_features(self, df):
        # Placeholder returning the input
        return df
        
    def predict_game_scores(self, features_df):
        # Mock prediction
        if features_df.empty:
            return pd.DataFrame()
            
        result = features_df.copy()
        result['predicted_home_final'] = 100
        result['predicted_away_final'] = 95
        return result
        
    def run_full_prediction_pipeline(self):
        # Mock prediction results
        return pd.DataFrame({
            'game_id': [1001, 1002],
            'home_team': ['Boston', 'LA Lakers'],
            'away_team': ['Philadelphia', 'Phoenix'],
            'current_quarter': [2, 3],
            'predicted_home_final': [105.5, 110.3],
            'predicted_away_final': [98.2, 103.7],
            'current_quarter': [2, 3]
        })

# Start with just testing the class initialization
monitor = NBAGameMonitor(update_interval=30, auto_save=False)
print("Monitor initialized successfully")

# Instead of running actual monitoring, let's just verify the logger works
monitor.log("Test log message")

In [ ]:
# Cell 8 - Charts begin

import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import time
import joblib
from IPython.display import clear_output
import pandas as pd

# Load the score prediction model
if 'score_model' not in globals():
    try:
        score_model = joblib.load(config.MODEL_PATH)
        print(f"Score prediction model loaded from {config.MODEL_PATH}")
    except Exception as e:
        print(f"Error loading model: {e}")
        score_model = None

# Define expected feature order
if 'expected_features' not in globals():
    expected_features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
    ]

# Initialize prediction history dictionary if it doesn't exist
if 'prediction_history' not in globals():
    prediction_history = {}

In [ ]:
# Cell 9: Additional Helper Functions for Enhanced Predictions

import math
from datetime import datetime, timedelta

def calculate_prediction_confidence(current_quarter: int) -> int:
    """Calculates confidence percentage based on game quarter."""
    confidence_map = {0: 30, 1: 45, 2: 65, 3: 80, 4: 95}
    return confidence_map.get(current_quarter, 30)

def get_season_scoring_adjustment() -> float:
    """Calculates a season scoring adjustment factor."""
    current_year = datetime.now().year
    if datetime.now().month >= 10:
        season = f"{current_year}-{current_year+1}"
    else:
        season = f"{current_year-1}-{current_year}"
    try:
        current_resp = supabase.table("nba_historical_game_stats").select("home_score,away_score").gte("game_date", f"{current_year-1}-10-01").execute()
        if not current_resp.data:
            print("No current season data found. Using default adjustment factor.")
            return 1.0
        current_df = pd.DataFrame(current_resp.data)
        current_avg = (current_df['home_score'].mean() + current_df['away_score'].mean()) / 2
        hist_resp = supabase.table("nba_historical_game_stats").select("home_score,away_score").lt("game_date", f"{current_year-1}-10-01").gte("game_date", f"{current_year-3}-10-01").execute()
        if not hist_resp.data:
            print("No historical data found. Using default adjustment factor.")
            return 1.0
        hist_df = pd.DataFrame(hist_resp.data)
        hist_avg = (hist_df['home_score'].mean() + hist_df['away_score'].mean()) / 2
        adjustment = current_avg / hist_avg if hist_avg > 0 else 1.0
        print(f"Season scoring adjustment: {adjustment:.3f} (Current: {current_avg:.1f}, Historical: {hist_avg:.1f})")
        return adjustment
    except Exception as e:
        print(f"Error calculating season adjustment: {e}")
        return 1.0

def calculate_win_probability(home_score: float, away_score: float, quarter: int, time_remaining: float = None) -> float:
    """Same as in Cell 7."""
    score_diff = home_score - away_score
    if time_remaining is not None:
        total_time = 48.0
        elapsed = total_time - time_remaining
        progress = elapsed / total_time
    else:
        progress = min(quarter / 4.0, 1.0)
    k = 0.05 + (progress * 0.15)
    return 1.0 / (1.0 + math.exp(-k * score_diff))


In [ ]:
# Cell 10: Data Fetching Functions

def get_team_rolling_averages(days_lookback: int = 60) -> dict:
    try:
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        if not response.data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return {}
        df = pd.DataFrame(response.data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df = df.sort_values('game_date')
        team_avgs = {}
        all_teams = set(df['home_team'].unique()).union(set(df['away_team'].unique()))
        for team in all_teams:
            home = df[df['home_team'] == team]['home_score']
            away = df[df['away_team'] == team]['away_score']
            all_scores = pd.concat([home, away])
            team_avgs[team] = all_scores.mean() if not all_scores.empty else 105.0
        return team_avgs
    except Exception as e:
        print(f"Error getting team rolling averages: {e}")
        traceback.print_exc()
        return {}

def get_previous_matchup_diff(home_team: str, away_team: str, max_lookback: int = 5) -> float:
    try:
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team).eq("away_team", away_team)\
            .order('game_date', desc=True).limit(max_lookback).execute()
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team).eq("away_team", home_team)\
            .order('game_date', desc=True).limit(max_lookback).execute()
        matchups = (response_home.data or []) + (response_away.data or [])
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
            diffs = []
            for game in matchups:
                if game['home_team'] == home_team and game['away_team'] == away_team:
                    diffs.append(game['home_score'] - game['away_score'])
                elif game['home_team'] == away_team and game['away_team'] == home_team:
                    diffs.append(game['away_score'] - game['home_score'])
            return np.mean(diffs) if diffs else 0
        return 0
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0


In [ ]:
# Cell 11: Improved score prediction function with enhanced data handling

from datetime import datetime
import pandas as pd
import numpy as np
import traceback

def predict_final_scores(live_games_df, team_avgs):
    """
    Predicts final scores for live games with enhanced feature support and
    improved data safety checks.
    
    Args:
        live_games_df: DataFrame with live game data
        team_avgs: Dictionary mapping team names to their scoring averages
        
    Returns:
        DataFrame with prediction results
    """
    results = []
    
    # Validate inputs
    if live_games_df is None or live_games_df.empty:
        print("No live games provided for prediction")
        return pd.DataFrame()
    
    if team_avgs is None or not team_avgs:
        print("Warning: No team averages provided, using default values")
        team_avgs = {}
    
    # Check that the game matchups are consistent with official schedule
    if 'official_schedule' in globals() and not globals()['official_schedule'].empty:
        schedule_df = globals()['official_schedule']
        # Filter to only include games in the official schedule
        valid_games = []
        for _, game in live_games_df.iterrows():
            home_team = game['home_team']
            away_team = game['away_team']
            # Check if this game is in the schedule
            if ((schedule_df['home_team'] == home_team) & 
                (schedule_df['away_team'] == away_team)).any():
                valid_games.append(game)
        
        if valid_games:
            live_games_df = pd.DataFrame(valid_games)
            print(f"Filtered to {len(live_games_df)} valid games in the official schedule")
    
    # Make a copy to avoid modifying the original
    live_games_df = live_games_df.copy()
    
    # Determine model type based on global model
    enhanced_model = False
    prediction_model = None
    
    if 'model' in globals() and globals()['model'] is not None:
        prediction_model = globals()['model']
        if hasattr(prediction_model, 'feature_importances_'):
            if len(prediction_model.feature_importances_) > 8:
                enhanced_model = True
    elif 'score_model' in globals() and globals()['score_model'] is not None:
        prediction_model = globals()['score_model']
        if hasattr(prediction_model, 'feature_importances_'):
            if len(prediction_model.feature_importances_) > 8:
                enhanced_model = True
    
    # Define expected features based on model type
    if enhanced_model:
        expected_features = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'prev_matchup_diff',
            'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]
        print("Using enhanced feature set for predictions")
    else:
        expected_features = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
        ]
        print("Using original feature set for predictions")
    
    print(f"Predicting scores for {len(live_games_df)} games")
    
    # Apply data preprocessing to all games at once
    try:
        # Clean and prepare game data
        live_games_df = calculate_derived_features(live_games_df)
        features_df = prepare_features_for_model(live_games_df, expected_features, team_avgs)
        print("Data preparation completed successfully")
    except Exception as e:
        print(f"Error in batch data preparation: {e}")
        traceback.print_exc()
        # Continue processing individual games if batch processing fails
        features_df = live_games_df.copy()
    
    # Process each game
    for idx, game in live_games_df.iterrows():
        try:
            game_id = game.get('game_id')
            home_team = game.get('home_team')
            away_team = game.get('away_team')
            
            if pd.isna(game_id) or pd.isna(home_team) or pd.isna(away_team):
                print(f"Skipping game with missing essential data (row {idx})")
                continue
                
            print(f"Processing game {game_id}: {home_team} vs {away_team}")
            
            # Extract current game state
            current_home_score = float(live_games_df.at[idx, 'home_score'] if pd.notna(live_games_df.at[idx, 'home_score']) else 0)
            current_away_score = float(live_games_df.at[idx, 'away_score'] if pd.notna(live_games_df.at[idx, 'away_score']) else 0)
            current_quarter = int(live_games_df.at[idx, 'current_quarter'] if pd.notna(live_games_df.at[idx, 'current_quarter']) else 0)
            
            # Get previous matchup difference for away score adjustment
            prev_matchup_diff = get_previous_matchup_diff(home_team, away_team)
            
            # Model-based prediction
            if prediction_model is not None:
                try:
                    # Get features for this specific game
                    if idx in features_df.index:
                        game_features = features_df.loc[[idx]]
                    else:
                        # Create features if not found in prepared dataframe
                        game_features = prepare_features_for_model(
                            pd.DataFrame([game]), expected_features, team_avgs)
                    
                    # Select only the expected feature columns in the right order
                    X = game_features[expected_features]
                    
                    # Generate prediction
                    predicted_home_score = prediction_model.predict(X)[0]
                    
                    # Calculate away score
                    diff_weight = min(0.3 + (0.1 * current_quarter), 0.6)
                    momentum_adj = float(game_features.get('cumulative_momentum', 0).iloc[0]) * 3.0 if enhanced_model else 0
                    predicted_away_score = predicted_home_score - (prev_matchup_diff * diff_weight) - momentum_adj
                    
                except Exception as e:
                    print(f"Error in model prediction: {e}")
                    # Fall back to simpler prediction
                    home_avg = team_avgs.get(home_team, 110.0)
                    away_avg = team_avgs.get(away_team, 110.0)
                    remaining_pct = 1.0 - (0.25 * current_quarter)
                    predicted_home_score = current_home_score + (home_avg * remaining_pct)
                    predicted_away_score = current_away_score + (away_avg * remaining_pct)
            else:
                # No model available, use simple extrapolation
                print("No prediction model available, using simple extrapolation")
                home_avg = team_avgs.get(home_team, 110.0)
                away_avg = team_avgs.get(away_team, 110.0)
                remaining_pct = 1.0 - (0.25 * current_quarter)
                predicted_home_score = current_home_score + (home_avg * remaining_pct)
                predicted_away_score = current_away_score + (away_avg * remaining_pct)
            
            # CRITICAL FIX: Ensure predicted scores are never less than current scores
            predicted_home_score = max(predicted_home_score, current_home_score)
            predicted_away_score = max(predicted_away_score, current_away_score)
            
            # Apply quarter-based adjustment
            quarter_weight = min(0.15 * current_quarter, 0.65)
            adjusted_home_score = (1 - quarter_weight) * predicted_home_score + quarter_weight * current_home_score
            adjusted_away_score = (1 - quarter_weight) * predicted_away_score + quarter_weight * current_away_score
            
            # Final sanity check - ensure all scores are at least current scores
            predicted_home_score = max(adjusted_home_score, current_home_score)
            predicted_away_score = max(adjusted_away_score, current_away_score)
            
            # Calculate confidence based on quarter
            confidence = calculate_prediction_confidence(current_quarter)
            
            # Calculate remaining points
            remaining_home = predicted_home_score - current_home_score
            remaining_away = predicted_away_score - current_away_score
            
            # Calculate win probability
            win_prob = calculate_win_probability(predicted_home_score, predicted_away_score, current_quarter)
            
            # Get momentum shift value
            momentum_shift = 0
            if 'cumulative_momentum' in features_df.columns and idx in features_df.index:
                momentum_shift = float(features_df.at[idx, 'cumulative_momentum'])
            
            # Calculate other metrics for dynamic recommendations
            projected_margin = predicted_home_score - predicted_away_score
            total_projected_score = predicted_home_score + predicted_away_score
            
            # Store result
            result = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'current_quarter': current_quarter,
                'current_home_score': current_home_score,
                'current_away_score': current_away_score,
                'predicted_home_final': predicted_home_score,
                'predicted_away_final': predicted_away_score,
                'remaining_home_points': remaining_home,
                'remaining_away_points': remaining_away,
                'confidence': confidence,
                'win_probability': win_prob,
                'momentum_shift': momentum_shift,
                'projected_margin': projected_margin,
                'total_projected_score': total_projected_score,
                'time_remaining': 12 * (4 - current_quarter) if current_quarter <= 4 else 0,
                'timestamp': datetime.now(),
                'is_fallback': False
            }
            results.append(result)
            
            # Update prediction history
            if 'prediction_history' in globals():
                if game_id not in globals()['prediction_history']:
                    globals()['prediction_history'][game_id] = []
                globals()['prediction_history'][game_id].append(result)
            
        except Exception as e:
            print(f"Error processing game {idx}: {e}")
            traceback.print_exc()
            # Try to extract basic info for logging
            try:
                game_id = game.get('game_id', f"unknown_{idx}")
                home_team = game.get('home_team', "Unknown Home")
                away_team = game.get('away_team', "Unknown Away")
                
                print(f"Adding fallback prediction for {home_team} vs {away_team}")
                
                # Create a minimal fallback result
                result = {
                    'game_id': game_id,
                    'home_team': home_team,
                    'away_team': away_team,
                    'current_quarter': 1,
                    'current_home_score': 0,
                    'current_away_score': 0,
                    'predicted_home_final': 105,
                    'predicted_away_final': 102,
                    'remaining_home_points': 105,
                    'remaining_away_points': 102,
                    'confidence': 25,  # Low confidence for fallback
                    'win_probability': 0.55,
                    'momentum_shift': 0,
                    'projected_margin': 3,
                    'total_projected_score': 207,
                    'time_remaining': 48,
                    'timestamp': datetime.now(),
                    'is_fallback': True
                }
                results.append(result)
            except:
                print("Could not create fallback prediction")
    
    # Final check - ensure we have results
    print(f"Generated {len(results)} predictions from {len(live_games_df)} games")
    return pd.DataFrame(results) if results else pd.DataFrame()

In [ ]:
# Cell 12: Updated Dashboard with Confidence Display

from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import pandas as pd

def create_live_dashboard(team_predictions: pd.DataFrame):
    """Creates a dashboard visualization of live game predictions with confidence metrics."""
    clear_output(wait=True)
    if team_predictions.empty:
        print("No live games to display.")
        return
    n_games = len(team_predictions)
    fig, axs = plt.subplots(n_games, 2, figsize=(15, 5*n_games))
    fig.suptitle(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", fontsize=16)
    if n_games == 1:
        axs = np.array([axs])
    for i, (_, game) in enumerate(team_predictions.iterrows()):
        ax_scores = axs[i, 0]
        teams = [game['home_team'], game['away_team']]
        current_scores = [game['current_home_score'], game['current_away_score']]
        predicted_scores = [game['predicted_home_score'], game['predicted_away_score']]
        x = np.arange(len(teams))
        width = 0.35
        ax_scores.bar(x - width/2, current_scores, width, label='Current')
        ax_scores.bar(x + width/2, predicted_scores, width, label='Predicted Final')
        ax_scores.set_xticks(x)
        ax_scores.set_xticklabels(teams)
        ax_scores.legend()
        confidence = game.get('prediction_confidence', 50)
        ax_scores.set_title(f"{game['home_team']} vs {game['away_team']} - Q{game['current_quarter']} (Confidence: {confidence}%)")
        for j, v in enumerate(current_scores):
            ax_scores.text(j - width/2, v + 1, str(int(v)), ha='center')
        for j, v in enumerate(predicted_scores):
            ax_scores.text(j + width/2, v + 1, f"{v:.1f}", ha='center')
        ax_history = axs[i, 1]
        game_id = game['game_id']
        if game_id in prediction_history and len(prediction_history[game_id]) > 1:
            history = pd.DataFrame(prediction_history[game_id])
            ax_history.plot(history['timestamp'], history['predicted_home_score'], label=f"{game['home_team']} Final", marker='o')
            ax_history.plot(history['timestamp'], history['predicted_away_score'], label=f"{game['away_team']} Final", marker='s')
            ax_history.set_title("Prediction Evolution")
            ax_history.set_xlabel("Time")
            ax_history.set_ylabel("Predicted Score")
            ax_history.legend()
            ax_history.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%H:%M'))
            plt.setp(ax_history.xaxis.get_majorticklabels(), rotation=45)
        else:
            ax_history.text(0.5, 0.5, "Not enough prediction history yet", ha='center', va='center', transform=ax_history.transAxes)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    print(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    for _, game in team_predictions.iterrows():
        quarter_text = "Pre-game" if game['current_quarter'] == 0 else f"Quarter {game['current_quarter']}"
        print(f"\n{game['home_team']} vs {game['away_team']} - {quarter_text}")
        print(f"Current Score: {game['home_team']} {game['current_home_score']} - {game['away_team']} {game['current_away_score']}")
        print(f"Predicted Final: {game['home_team']} {game['predicted_home_score']:.1f} - {game['away_team']} {game['predicted_away_score']:.1f}")
        if 'win_probability' in game:
            print(f"Win Probability: {game['win_probability']*100:.1f}%")
        print("-"*80)

# Call this function with your prediction DataFrame to display the dashboard.


In [ ]:
# Cell 13: Enhanced Model Validation for Enhanced Features

def validate_model_on_historical_games(num_games: int = 10):
    """Validates the prediction model using historical games and displays error metrics."""
    print("Starting validation on historical games...")
    try:
        model_to_use = model if 'model' in globals() and model is not None else None
        if model_to_use is None:
            print("No prediction model loaded. Skipping validation.")
            return None
        response = supabase.table("nba_historical_game_stats").select("*").order('game_date', desc=True).limit(num_games).execute()
        if not response.data:
            print("No historical games found for validation.")
            return None
        historical_games = response.data
        team_avgs = get_team_rolling_averages()
        validation_results = []
        print(f"Validating model on {len(historical_games)} games")
        for game in historical_games:
            actual_home = game['home_score']
            actual_away = game['away_score']
            game_id = game['game_id']
            for quarter in range(1, 5):
                sim_game = {
                    'game_id': game_id,
                    'home_team': game['home_team'],
                    'away_team': game['away_team'],
                    'game_date': game.get('game_date')
                }
                for q in range(1, 5):
                    sim_game[f'home_q{q}'] = game.get(f'home_q{q}', 0) if q <= quarter else 0
                    sim_game[f'away_q{q}'] = game.get(f'away_q{q}', 0) if q <= quarter else 0
                sim_game['home_score'] = sum([sim_game[f'home_q{q}'] for q in range(1, quarter+1)])
                sim_game['away_score'] = sum([sim_game[f'away_q{q}'] for q in range(1, quarter+1)])
                temp_df = pd.DataFrame([sim_game])
                pred_result = predict_final_scores(temp_df, team_avgs)
                if pred_result.empty:
                    continue
                pred_row = pred_result.iloc[0]
                home_error = pred_row['predicted_home_score'] - actual_home
                away_error = pred_row['predicted_away_score'] - actual_away
                validation_results.append({
                    'game_id': game_id,
                    'quarter': quarter,
                    'actual_home': actual_home,
                    'actual_away': actual_away,
                    'predicted_home': pred_row['predicted_home_score'],
                    'predicted_away': pred_row['predicted_away_score'],
                    'home_error': home_error,
                    'away_error': away_error,
                    'absolute_error': (abs(home_error) + abs(away_error)) / 2
                })
        if not validation_results:
            print("No validation results generated – check that games have quarter data.")
            return None
        results_df = pd.DataFrame(validation_results)
        print("\nValidation Results:")
        metrics_by_quarter = results_df.groupby('quarter').agg({
            'home_error': 'mean',
            'away_error': 'mean',
            'absolute_error': 'mean'
        })
        print(metrics_by_quarter)
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        metrics_by_quarter[['absolute_error']].plot(kind='bar', title='Mean Absolute Error by Quarter', legend=False)
        plt.ylabel('Error (points)')
        plt.subplot(1, 2, 2)
        results_df.boxplot(column=['home_error', 'away_error'], by='quarter')
        plt.title('Error Distribution by Quarter')
        plt.suptitle('')
        plt.tight_layout()
        plt.show()
        return results_df
    except Exception as e:
        print(f"Error during validation: {e}")
        traceback.print_exc()
        return None

# Optionally call validation function:
# validation_df = validate_model_on_historical_games(10)


In [ ]:
# Cell 14: Fetch Schedule Data

def fetch_scheduled_games(date_str: str = None) -> pd.DataFrame:
    """Fetch scheduled games from the database for a specific date (Pacific Time)."""
    import pytz
    if date_str is None:
        pacific_tz = pytz.timezone('America/Los_Angeles')
        date_str = datetime.now(pacific_tz).strftime('%Y-%m-%d')
    print(f"Fetching scheduled games for {date_str} (Pacific Time)")
    try:
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute()
        if response.data:
            return pd.DataFrame(response.data)
        else:
            print(f"No scheduled games found for {date_str}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error fetching scheduled games: {e}")
        return pd.DataFrame()

today_schedule = fetch_scheduled_games()
display(today_schedule)


In [ ]:
# Cell 14B - Add explicit game status tracking with Pacific Time Zone support

import pytz
from datetime import datetime, timedelta

def determine_game_status(games_df):
    """
    Adds explicit status field to game data with proper Pacific Time Zone handling:
    - LIVE: Currently in progress
    - SCHEDULED: Upcoming game
    - FINAL: Completed game
    
    Also adds estimated time remaining.
    """
    if games_df is None or games_df.empty:
        return games_df
        
    result_df = games_df.copy()
    result_df['game_status'] = 'UNKNOWN'
    result_df['time_remaining_mins'] = 0
    
    # Current time in Pacific Time Zone (Los Angeles)
    pacific_tz = pytz.timezone('America/Los_Angeles')
    now_pacific = datetime.now(pacific_tz)
    today_pacific = now_pacific.strftime('%Y-%m-%d')
    
    print(f"Current Pacific Time: {now_pacific.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    
    for idx, game in result_df.iterrows():
        # Extract game info
        game_date = game.get('game_date', today_pacific)
        
        # Calculate quarter based on data
        current_q = 0
        for q in range(1, 5):
            home_q_val = float(game.get(f'home_q{q}', 0) or 0)
            away_q_val = float(game.get(f'away_q{q}', 0) or 0)
            if home_q_val > 0 or away_q_val > 0:
                current_q = q
        
        # Set current_quarter field consistent with data
        result_df.at[idx, 'current_quarter'] = current_q
        
        # Convert game_date to datetime object in Pacific time
        try:
            if isinstance(game_date, str):
                # If time component is missing, assume start of day
                if len(game_date) <= 10:  # YYYY-MM-DD format
                    game_date_obj = datetime.strptime(game_date, '%Y-%m-%d')
                    # Add typical NBA game start time (7:30 PM Pacific)
                    game_date_obj = game_date_obj.replace(hour=19, minute=30)
                    # Localize to Pacific time
                    game_date_obj = pacific_tz.localize(game_date_obj)
                else:
                    # Parse with time component
                    game_date_obj = pd.to_datetime(game_date).tz_localize(pacific_tz)
            else:
                # Handle if already datetime object but no timezone
                game_date_obj = pd.to_datetime(game_date)
                if game_date_obj.tzinfo is None:
                    game_date_obj = game_date_obj.tz_localize(pacific_tz)
        except Exception as e:
            print(f"Error parsing game date '{game_date}': {e}")
            game_date_obj = now_pacific
        
        # Determine status based on available data and Pacific time
        if current_q > 0 and current_q < 4:
            # Game in progress
            result_df.at[idx, 'game_status'] = 'LIVE'
            result_df.at[idx, 'time_remaining_mins'] = (4 - current_q) * 12
        elif current_q == 4:
            # Potentially finished or in 4th quarter
            home_q4 = float(game.get('home_q4', 0) or 0)
            away_q4 = float(game.get('away_q4', 0) or 0)
            if home_q4 > 0 and away_q4 > 0:
                # Check if complete or still in Q4
                if game.get('is_finished', False):
                    result_df.at[idx, 'game_status'] = 'FINAL'
                    result_df.at[idx, 'time_remaining_mins'] = 0
                else:
                    result_df.at[idx, 'game_status'] = 'LIVE'
                    result_df.at[idx, 'time_remaining_mins'] = 6  # Estimate mid-Q4
            else:
                result_df.at[idx, 'game_status'] = 'LIVE'
                result_df.at[idx, 'time_remaining_mins'] = 12  # Full Q4 remaining
        elif current_q == 0:
            # Game hasn't started yet - check date against Pacific time
            if game_date_obj.date() < now_pacific.date():
                # Past date but no scores - likely a data issue or postponed game
                result_df.at[idx, 'game_status'] = 'UNKNOWN'
            elif game_date_obj.date() == now_pacific.date():
                # Today's game
                if game_date_obj < now_pacific:
                    # Start time has passed, but no scores - may be delayed or data issue
                    result_df.at[idx, 'game_status'] = 'DELAYED'
                else:
                    # Start time is later today
                    result_df.at[idx, 'game_status'] = 'SCHEDULED'
                    # Calculate minutes until game starts
                    mins_to_start = (game_date_obj - now_pacific).total_seconds() / 60
                    result_df.at[idx, 'time_remaining_mins'] = 48 + max(0, mins_to_start)
            else:
                # Future date
                result_df.at[idx, 'game_status'] = 'SCHEDULED'
                result_df.at[idx, 'time_remaining_mins'] = 48  # Full game
        
        # Special case: Check for actual final flag (for historical test games)
        if 'actual_home_final' in game and pd.notna(game['actual_home_final']):
            result_df.at[idx, 'game_status'] = 'FINAL'
            result_df.at[idx, 'time_remaining_mins'] = 0
    
    # Summary of statuses
    status_counts = result_df['game_status'].value_counts()
    print("Game status summary:")
    for status, count in status_counts.items():
        print(f"  - {status}: {count} games")
    
    return result_df

In [ ]:
# Cell 15: Enhanced Monitoring Function with Correct Team Matching

from models.dynamic_recommendation import generate_recommendations
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import traceback
import pytz
import os
import joblib
from supabase import create_client

# Make the official schedule globally available
OFFICIAL_SCHEDULE = None

def normalize_team_name(name):
    """Normalize team names for consistent matching"""
    if not name:
        return ""

    # Convert to string and lowercase
    name = str(name).lower().strip()

    # Standard team name mappings
    mappings = {
        'lakers': 'los angeles lakers',
        'la lakers': 'los angeles lakers',
        'clippers': 'los angeles clippers',
        'la clippers': 'los angeles clippers',
        'blazers': 'portland trail blazers',
        'sixers': 'philadelphia 76ers',
        'philly': 'philadelphia 76ers',
        'knicks': 'new york knicks',
        'ny knicks': 'new york knicks',
        'nets': 'brooklyn nets',
        'mavs': 'dallas mavericks',
        'cavs': 'cleveland cavaliers',
        'wolves': 'minnesota timberwolves',
        't-wolves': 'minnesota timberwolves',
        'celts': 'boston celtics',
        'pels': 'new orleans pelicans',
        'warriors': 'golden state warriors',
        'gsw': 'golden state warriors',
        'heat': 'miami heat',
        'bulls': 'chicago bulls',
        'hawks': 'atlanta hawks',
        'suns': 'phoenix suns',
        'bucks': 'milwaukee bucks',
        'jazz': 'utah jazz',
        'nuggets': 'denver nuggets',
        'rockets': 'houston rockets',
        'pacers': 'indiana pacers',
        'spurs': 'san antonio spurs',
        'kings': 'sacramento kings',
        'magic': 'orlando magic',
        'wizards': 'washington wizards',
        'raptors': 'toronto raptors',
        'thunder': 'oklahoma city thunder',
        'okc': 'oklahoma city thunder',
        'pistons': 'detroit pistons',
        'hornets': 'charlotte hornets',
        'grizzlies': 'memphis grizzlies',
        'grizz': 'memphis grizzlies'
    }

    # Check if the name is in mappings
    for key, value in mappings.items():
        if key in name:
            return value

    return name

def load_and_cache_official_schedule():
    """Loads and caches the official NBA schedule for today"""
    # Get today's date in Pacific Time (NBA standard timezone)
    pacific_tz = pytz.timezone('America/Los_Angeles')
    today = datetime.now(pacific_tz).strftime('%Y-%m-%d')

    try:
        # Try to fetch from database
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", today).execute()

        if response.data:
            schedule_df = pd.DataFrame(response.data)
            print(f"Loaded {len(schedule_df)} games from official schedule for {today}")

            # Add normalized team names for better matching
            schedule_df['home_team_norm'] = schedule_df['home_team'].apply(normalize_team_name)
            schedule_df['away_team_norm'] = schedule_df['away_team'].apply(normalize_team_name)

            global OFFICIAL_SCHEDULE
            OFFICIAL_SCHEDULE = schedule_df
            
            # Print the matchups
            for _, game in schedule_df.iterrows():
                print(f"  - {game['home_team']} vs {game['away_team']}")
            
            return schedule_df
        else:
            print(f"No scheduled games found for today ({today})")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error loading official schedule: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def fetch_live_games_with_schedule_matching():
    """Fetch live games and match against the official schedule"""
    # First load the official schedule
    global OFFICIAL_SCHEDULE
    if OFFICIAL_SCHEDULE is None or OFFICIAL_SCHEDULE.empty:
        OFFICIAL_SCHEDULE = load_and_cache_official_schedule()

    try:
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone('America/Los_Angeles')
        today_pacific = datetime.now(pacific_tz).strftime('%Y-%m-%d')

        print(f"Fetching live games for {today_pacific} (Pacific Time)")

        response = supabase.table("nba_live_game_stats").select("*").execute()
        
        if not response.data:
            print(f"No live games found. Checking for scheduled games...")
            
            # If there are scheduled games but no live games, convert schedule to live format
            if not OFFICIAL_SCHEDULE.empty:
                print("No live games found, but scheduled games exist. Creating pre-game entries.")
                live_games_df = convert_scheduled_to_live_format(OFFICIAL_SCHEDULE)
                return live_games_df
            else:
                print("No live or scheduled games found.")
                return pd.DataFrame()

        live_df = pd.DataFrame(response.data)
        
        # Filter for today's games if we have a date column
        if 'game_date' in live_df.columns:
            # Convert to datetime to handle different date formats
            live_df['game_date'] = pd.to_datetime(live_df['game_date'])
            today_dt = pd.to_datetime(today_pacific)
            live_df = live_df[live_df['game_date'].dt.date == today_dt.date()]
        
        print(f"Found {len(live_df)} live game records")

        # Validate and clean game data
        live_df = validate_game_data(live_df)

        # Match against the schedule if available
        if not OFFICIAL_SCHEDULE.empty:
            matched_df = match_teams_to_schedule(live_df, OFFICIAL_SCHEDULE)
            print(f"Matched {matched_df['matched_to_schedule'].sum()} of {len(matched_df)} games to schedule")
            return matched_df
        else:
            # If no schedule, just return the validated live data
            return live_df

    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()

        # If live data fails but we have the schedule, use that
        if not OFFICIAL_SCHEDULE.empty:
            print("Using schedule data as fallback")
            return convert_scheduled_to_live_format(OFFICIAL_SCHEDULE)

        return pd.DataFrame()

def find_recent_games_for_testing():
    """Find recent completed games to use for testing the prediction model"""
    print("No live games found. Fetching recent completed games for testing...")

    # Get dates to try, in order of preference
    try:
        dates_to_try = []
        today = datetime.now()

        # Try yesterday first, then previous days
        for i in range(1, 5):  # Try up to 4 previous days
            date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
            dates_to_try.append(date)

        # Try each date in sequence
        for test_date in dates_to_try:
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .eq("game_date", test_date)\
                .limit(5).execute()

            historical_games = response.data
            if historical_games:
                print(f"Found {len(historical_games)} games from {test_date} for testing.")

                # Simulate these as 'live' games by setting them to random quarters
                import random

                live_games = []
                for game in historical_games:
                    # Randomly select a quarter for simulation (1-4)
                    sim_quarter = random.randint(1, 4)

                    # Create a simulated live game where we only know scores up to the simulated quarter
                    sim_game = {
                        'game_id': game['game_id'],
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': sim_quarter,
                        'simulated_quarter': sim_quarter,  # Mark which quarter we're simulating
                        'matched_to_schedule': True  # Pretend it's matched
                    }

                    # Add quarter scores up to the simulated quarter
                    for q in range(1, 5):
                        q_field_home = f'home_q{q}'
                        q_field_away = f'away_q{q}'

                        if q <= sim_quarter and q_field_home in game and q_field_away in game:
                            sim_game[q_field_home] = game[q_field_home]
                            sim_game[q_field_away] = game[q_field_away]
                        else:
                            sim_game[q_field_home] = 0
                            sim_game[q_field_away] = 0

                    # Calculate the current score based on quarters we "know"
                    sim_game['home_score'] = sum([
                        float(sim_game.get(f'home_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                    ])
                    sim_game['away_score'] = sum([
                        float(sim_game.get(f'away_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                    ])

                    # Save the actual final scores for validation
                    sim_game['actual_final_home'] = game['home_score']
                    sim_game['actual_final_away'] = game['away_score']

                    live_games.append(sim_game)

                return pd.DataFrame(live_games)

        print("No recent games found for testing.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error finding recent games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def get_team_rolling_averages(days_lookback=60):
    """
    Retrieves the rolling scoring average for each team from historical data.
    
    Args:
        days_lookback: Number of days to look back for calculating the average (default: 60)
        
    Returns:
        Dictionary mapping team names to their rolling scoring average
    """
    try:
        # Calculate the date threshold
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        
        # Fetch recent historical game data
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        historical_data = response.data
        
        if not historical_data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return {}
        
        df = pd.DataFrame(historical_data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df = df.sort_values('game_date')
        
        # Initialize dictionary for team averages
        team_avgs = {}
        
        # Get unique teams
        all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
        
        for team in all_teams:
            # Get home games where team is home
            home_games = df[df['home_team'] == team][['game_date', 'home_score']].rename(
                columns={'home_score': 'score'})
            
            # Get away games where team is away
            away_games = df[df['away_team'] == team][['game_date', 'away_score']].rename(
                columns={'away_score': 'score'})
            
            # Combine all games
            team_games = pd.concat([home_games, away_games]).sort_values('game_date')
            
            if not team_games.empty:
                # Calculate recent average (last 5 games if available)
                recent_games = team_games.tail(5)
                team_avgs[team] = recent_games['score'].mean()
            else:
                # Fallback to a reasonable default
                team_avgs[team] = 105.0  # NBA average is approximately 100-110 points per game
        
        return team_avgs
    except Exception as e:
        print(f"Error getting team rolling averages: {e}")
        return {}  # Return empty dict on error

def get_rest_data(team_name, game_date):
    """
    Calculates rest days for a team before a specific game date.
    
    Args:
        team_name: Name of the team
        game_date: Date of the current game
        
    Returns:
        Dictionary with rest days and back-to-back status
    """
    try:
        # Ensure game_date is datetime
        if isinstance(game_date, str):
            game_date = pd.to_datetime(game_date)
        
        # Try to find previous game with a simple query approach
        try:
            # First attempt - check home games
            home_response = supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("home_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
                
            # Second attempt - check away games
            away_response = supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("away_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
            
            # Combine results and find most recent
            all_games = home_response.data + away_response.data
            if not all_games:
                # If no previous game found, return default
                return {
                    'rest_days': 3,  # Default if no history
                    'is_back_to_back': False
                }
                
            # Sort by date (most recent first)
            all_games.sort(key=lambda x: x['game_date'], reverse=True)
            prev_game_date = pd.to_datetime(all_games[0]['game_date'])
            
        except Exception as e:
            print(f"Error in first query approach: {e}")
            # Try with normalized team name as fallback
            norm_team = normalize_team_name(team_name)
            if norm_team != team_name:
                try:
                    # Look for the team's previous game with normalized name
                    home_response = supabase.table("nba_historical_game_stats")\
                        .select("game_date")\
                        .eq("home_team", norm_team)\
                        .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                        .order("game_date", desc=True)\
                        .limit(1)\
                        .execute()
                        
                    away_response = supabase.table("nba_historical_game_stats")\
                        .select("game_date")\
                        .eq("away_team", norm_team)\
                        .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                        .order("game_date", desc=True)\
                        .limit(1)\
                        .execute()
                    
                    all_games = home_response.data + away_response.data
                    if not all_games:
                        return {
                            'rest_days': 3,
                            'is_back_to_back': False
                        }
                        
                    all_games.sort(key=lambda x: x['game_date'], reverse=True)
                    prev_game_date = pd.to_datetime(all_games[0]['game_date'])
                except Exception as e2:
                    print(f"Error in fallback query: {e2}")
                    return {
                        'rest_days': 2,  # Default if all queries fail
                        'is_back_to_back': False
                    }
            else:
                return {
                    'rest_days': 2,
                    'is_back_to_back': False
                }
        
        # Calculate days between games
        rest_days = (game_date - prev_game_date).days
        
        # Cap to reasonable values
        rest_days = max(min(rest_days, 10), 0)
        
        return {
            'rest_days': rest_days,
            'is_back_to_back': rest_days <= 1
        }
    except Exception as e:
        print(f"Error getting rest data for {team_name}: {e}")
        # Return default values on error
        return {
            'rest_days': 2,  # Reasonable default
            'is_back_to_back': False
        }

def get_previous_matchup_diff(home_team, away_team, max_lookback=5):
    """Gets the point differential from previous matchups between two teams."""
    try:
        # Use separate queries for home and away configurations to avoid syntax issues
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        
        # Combine results
        home_matchups = response_home.data
        away_matchups = response_away.data
        matchups = home_matchups + away_matchups
        
        # Sort by date (most recent first)
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
        
        if not matchups:
            return 0
        
        # Calculate point differential from home team perspective
        differentials = []
        for game in matchups:
            if game['home_team'] == home_team and game['away_team'] == away_team:
                diff = game['home_score'] - game['away_score']
            elif game['home_team'] == away_team and game['away_team'] == home_team:
                diff = game['away_score'] - game['home_score']
            else:
                continue
            differentials.append(diff)
        
        # Cap extreme values
        avg_diff = sum(differentials) / len(differentials) if differentials else 0
        return max(min(avg_diff, 15), -15)  # Cap at +/- 15 points
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0

def convert_scheduled_to_live_format(schedule_df):
    """Convert scheduled games to the live game format for pre-game predictions"""
    live_format = []
    
    for _, game in schedule_df.iterrows():
        # Create a basic live game structure
        live_game = {
            'game_id': game['game_id'],
            'home_team': game['home_team'],
            'away_team': game['away_team'],
            'game_date': game['game_date'],
            'matched_to_schedule': True,
            'current_quarter': 0,  # Pre-game
            'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
            'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
            'home_score': 0,
            'away_score': 0
        }
        live_format.append(live_game)
    
    return pd.DataFrame(live_format)

def validate_game_data(df):
    """Validate and clean game data"""
    # Make a copy to avoid modifying the original
    df = df.copy()
    
    # Convert date fields to datetime
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    
    # Ensure quarter fields are numeric
    for q in range(1, 5):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in df.columns:
            df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
        if away_q in df.columns:
            df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
    
    # Calculate current quarter based on non-zero quarter scores
    df['current_quarter'] = 0
    for idx, row in df.iterrows():
        max_quarter = 0
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                max_quarter = q
        
        df.at[idx, 'current_quarter'] = max_quarter
    
    # Ensure team names are strings
    if 'home_team' in df.columns:
        df['home_team'] = df['home_team'].astype(str)
    if 'away_team' in df.columns:
        df['away_team'] = df['away_team'].astype(str)
    
    # Calculate current score as sum of quarters
    if 'home_score' not in df.columns:
        df['home_score'] = 0
    if 'away_score' not in df.columns:
        df['away_score'] = 0
        
    for idx, row in df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        
        home_score = 0
        away_score = 0
        for q in range(1, current_q + 1):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in row:
                home_score += float(row[home_q] or 0)
            if away_q in row:
                away_score += float(row[away_q] or 0)
        
        df.at[idx, 'home_score'] = home_score
        df.at[idx, 'away_score'] = away_score
    
    # Calculate score ratio
    df['score_ratio'] = 0.5  # Default to even
    for idx, row in df.iterrows():
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        total = home_score + away_score
        
        if total > 0:
            df.at[idx, 'score_ratio'] = home_score / total
    
    # Initialize matched_to_schedule column if not present
    if 'matched_to_schedule' not in df.columns:
        df['matched_to_schedule'] = False
    
    return df

def match_teams_to_schedule(live_df, schedule_df):
    """Match live game data to the official schedule"""
    # Add a column to track if games match the schedule
    live_df['matched_to_schedule'] = False
    
    # Copy to avoid modifying the original
    live_df = live_df.copy()
    
    # Add normalized team names for better matching
    live_df['home_team_norm'] = live_df['home_team'].apply(normalize_team_name)
    live_df['away_team_norm'] = live_df['away_team'].apply(normalize_team_name)
    
    # Make sure schedule has normalized teams too
    if 'home_team_norm' not in schedule_df.columns:
        schedule_df = schedule_df.copy()
        schedule_df['home_team_norm'] = schedule_df['home_team'].apply(normalize_team_name)
        schedule_df['away_team_norm'] = schedule_df['away_team'].apply(normalize_team_name)
    
    # Try to match by exact normalized team names
    for live_idx, live_game in live_df.iterrows():
        # Skip already matched games
        if live_game['matched_to_schedule']:
            continue
            
        home_norm = live_game['home_team_norm']
        away_norm = live_game['away_team_norm']
        
        # Try both home/away combinations
        home_away_match = schedule_df[
            (schedule_df['home_team_norm'] == home_norm) &
            (schedule_df['away_team_norm'] == away_norm)
        ]
        
        away_home_match = schedule_df[
            (schedule_df['home_team_norm'] == away_norm) &
            (schedule_df['away_team_norm'] == home_norm)
        ]
        
        if not home_away_match.empty:
            # Found exact match
            match = home_away_match.iloc[0]
            live_df.at[live_idx, 'matched_to_schedule'] = True
            # Copy over the official game_id if needed
            if 'game_id' in match and match['game_id'] != live_game['game_id']:
                live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                live_df.at[live_idx, 'game_id'] = match['game_id']
                
        elif not away_home_match.empty:
            # Found match with teams reversed - this indicates a data issue
            print(f"Warning: Found teams in reverse order for game {live_game['game_id']}")
            match = away_home_match.iloc[0]
            live_df.at[live_idx, 'matched_to_schedule'] = True
            live_df.at[live_idx, 'teams_reversed'] = True
            # Copy over the official game_id if needed
            if 'game_id' in match and match['game_id'] != live_game['game_id']:
                live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                live_df.at[live_idx, 'game_id'] = match['game_id']
    
    # For unmatched games, try looser matching
    if not live_df[~live_df['matched_to_schedule']].empty:
        for live_idx, live_game in live_df[~live_df['matched_to_schedule']].iterrows():
            best_match = None
            best_score = 0
            is_reversed = False
            
            for _, sched_game in schedule_df.iterrows():
                # Try both directions
                direct_match = (live_game['home_team_norm'] in sched_game['home_team_norm'] or 
                               sched_game['home_team_norm'] in live_game['home_team_norm']) and \
                              (live_game['away_team_norm'] in sched_game['away_team_norm'] or 
                               sched_game['away_team_norm'] in live_game['away_team_norm'])
                               
                reverse_match = (live_game['home_team_norm'] in sched_game['away_team_norm'] or 
                                sched_game['away_team_norm'] in live_game['home_team_norm']) and \
                               (live_game['away_team_norm'] in sched_game['home_team_norm'] or 
                                sched_game['home_team_norm'] in live_game['away_team_norm'])
                
                # Simple scoring - each match is worth 1 point
                score = direct_match + reverse_match * 0.9  # Slightly prefer direct matches
                
                if score > best_score:
                    best_score = score
                    best_match = sched_game
                    is_reversed = reverse_match > direct_match
            
            if best_match is not None and best_score > 0:
                live_df.at[live_idx, 'matched_to_schedule'] = True
                live_df.at[live_idx, 'match_confidence'] = best_score
                live_df.at[live_idx, 'teams_reversed'] = is_reversed
                
                # Copy over the official game_id if needed
                if 'game_id' in best_match and best_match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = best_match['game_id']
    
    return live_df

def run_enhanced_prediction_pipeline():
    """
    Run the full prediction pipeline with all improvements
    
    1. Ensure official schedule is loaded
    2. Fetch live games with schedule matching
    3. If no live games, use historical games for testing
    4. Calculate enhanced features
    5. Generate predictions using the appropriate model
    6. Calculate confidence metrics and win probabilities
    7. Return predictions with validation metrics when possible
    """
    # 1. Load official schedule
    global OFFICIAL_SCHEDULE
    if OFFICIAL_SCHEDULE is None:
        OFFICIAL_SCHEDULE = load_and_cache_official_schedule()

    # 2. Fetch live games
    live_games_df = fetch_live_games_with_schedule_matching()

    # 3. Fall back to historical games if needed
    if live_games_df.empty:
        live_games_df = find_recent_games_for_testing()

        if live_games_df.empty:
            print("No games available for prediction")
            return pd.DataFrame()

    # 4. Calculate enhanced features
    try:
        # Get team rolling averages
        team_avgs = get_team_rolling_averages()

        # Calculate rest features
        for idx, game in live_games_df.iterrows():
            try:
                home_team = game['home_team']
                away_team = game['away_team']
                game_date = pd.to_datetime(game['game_date'])

                # Get rest data for both teams
                home_rest = get_rest_data(home_team, game_date)
                away_rest = get_rest_data(away_team, game_date)

                # Update DataFrame
                live_games_df.at[idx, 'rest_days_home'] = home_rest['rest_days']
                live_games_df.at[idx, 'rest_days_away'] = away_rest['rest_days']
                live_games_df.at[idx, 'is_back_to_back_home'] = int(home_rest['is_back_to_back'])
                live_games_df.at[idx, 'is_back_to_back_away'] = int(away_rest['is_back_to_back'])
                live_games_df.at[idx, 'rest_advantage'] = home_rest['rest_days'] - away_rest['rest_days']

                # Get previous matchup difference
                live_games_df.at[idx, 'prev_matchup_diff'] = get_previous_matchup_diff(home_team, away_team)
            except Exception as e:
                print(f"Error calculating rest features for game {game.get('game_id')}: {e}")
                # Set default values
                live_games_df.at[idx, 'rest_days_home'] = 2
                live_games_df.at[idx, 'rest_days_away'] = 2
                live_games_df.at[idx, 'is_back_to_back_home'] = 0
                live_games_df.at[idx, 'is_back_to_back_away'] = 0
                live_games_df.at[idx, 'rest_advantage'] = 0
                live_games_df.at[idx, 'prev_matchup_diff'] = 0

        # Calculate momentum features
        for idx, game in live_games_df.iterrows():
            try:
                current_quarter = int(game.get('current_quarter', 0) or 0)

                # Initialize momentum features
                live_games_df.at[idx, 'q1_to_q2_momentum'] = 0
                live_games_df.at[idx, 'q2_to_q3_momentum'] = 0
                live_games_df.at[idx, 'q3_to_q4_momentum'] = 0
                live_games_df.at[idx, 'cumulative_momentum'] = 0

                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    # Q1 to Q2
                    home_q1 = float(game.get('home_q1', 0) or 0)
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    away_q1 = float(game.get('away_q1', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)

                    q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
                    # Cap to prevent extreme values
                    q1_to_q2 = max(min(q1_to_q2, 15), -15)
                    live_games_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2

                if current_quarter >= 3:
                    # Q2 to Q3
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)

                    q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
                    q2_to_q3 = max(min(q2_to_q3, 15), -15)
                    live_games_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3

                if current_quarter >= 4:
                    # Q3 to Q4
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    home_q4 = float(game.get('home_q4', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)
                    away_q4 = float(game.get('away_q4', 0) or 0)

                    q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
                    q3_to_q4 = max(min(q3_to_q4, 15), -15)
                    live_games_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4

                # Calculate weighted momentum
                weights = [0.2, 0.3, 0.5]  # Weights for each momentum period

                if current_quarter == 2:
                    momentum = live_games_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    momentum = (
                        live_games_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        live_games_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    momentum = (
                        live_games_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        live_games_df.at[idx, 'q2_to_q3_momentum'] * weights[1] +
                        live_games_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    momentum = 0

                # Normalize to [-1, 1]
                live_games_df.at[idx, 'cumulative_momentum'] = max(min(momentum / 15.0, 1.0), -1.0)

            except Exception as e:
                print(f"Error calculating momentum for game {game.get('game_id')}: {e}")
                # Set defaults
                live_games_df.at[idx, 'q1_to_q2_momentum'] = 0
                live_games_df.at[idx, 'q2_to_q3_momentum'] = 0
                live_games_df.at[idx, 'q3_to_q4_momentum'] = 0
                live_games_df.at[idx, 'cumulative_momentum'] = 0

    except Exception as e:
        print(f"Error calculating enhanced features: {e}")
        traceback.print_exc()
        # Continue with basic features

    # 5. Make predictions
    try:
        # Load model from global scope or load it if not present
        model_to_use = None
        if 'model' in globals() and globals()['model'] is not None:
            model_to_use = globals()['model']
            print("Using 'model' from global scope")
        else:
            # Try to load model from various locations
            try:
                model_path = config.MODEL_PATH
                if os.path.exists(model_path):
                    model_to_use = joblib.load(model_path)
                    print(f"Loaded model from {model_path}")
            except Exception as e:
                print(f"Failed to load model: {e}")

        if model_to_use is None:
            print("No model available for prediction")
            return live_games_df  # Return with features but no predictions

        # Determine feature list based on model
        if hasattr(model_to_use, 'feature_importances_'):
            feature_count = len(model_to_use.feature_importances_)
            if feature_count > 8:
                # Enhanced model
                features_to_use = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                print("Using enhanced feature set for prediction")
            else:
                # Original model
                features_to_use = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]

                # Add rolling scores if needed
                if 'rolling_home_score' not in live_games_df.columns:
                    live_games_df['rolling_home_score'] = live_games_df['home_team'].apply(
                        lambda t: team_avgs.get(t, 105.0))
                    live_games_df['rolling_away_score'] = live_games_df['away_team'].apply(
                        lambda t: team_avgs.get(t, 105.0))

                print("Using original feature set for prediction")
        else:
            # Default to enhanced features
            features_to_use = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
            print("Using default enhanced feature set")

        # Make sure all required features exist and have reasonable values
        for feature in features_to_use:
            if feature not in live_games_df.columns:
                print(f"Adding missing feature: {feature}")
                live_games_df[feature] = 0
            
            # Convert to numeric and handle NaN values
            live_games_df[feature] = pd.to_numeric(live_games_df[feature], errors='coerce').fillna(0)

        # Prepare feature matrix
        X_pred = live_games_df[features_to_use].copy()

        # Generate predictions
        predictions = model_to_use.predict(X_pred)
        live_games_df['predicted_home_score'] = predictions

        # Calculate additional metrics
        for idx, row in live_games_df.iterrows():
            # Estimate away score (if model only predicts home score)
            # Use simple home court advantage (avg ~3.5 points) and previous matchup differential
            home_advantage = 3.5
            matchup_diff = row.get('prev_matchup_diff', 0)
            live_games_df.at[idx, 'predicted_away_score'] = row['predicted_home_score'] - home_advantage - matchup_diff

            # Calculate remaining quarters
            current_quarter = int(row.get('current_quarter', 0) or 0)
            if current_quarter > 0 and current_quarter < 4:
                # Get current score
                current_home_score = sum([float(row.get(f'home_q{q}', 0) or 0) for q in range(1, current_quarter + 1)])
                current_away_score = sum([float(row.get(f'away_q{q}', 0) or 0) for q in range(1, current_quarter + 1)])
                
                # Calculate remaining score
                remaining_home = row['predicted_home_score'] - current_home_score
                remaining_away = row['predicted_away_score'] - current_away_score
                
                # Store remaining scores
                live_games_df.at[idx, 'remaining_home_score'] = max(0, remaining_home)
                live_games_df.at[idx, 'remaining_away_score'] = max(0, remaining_away)
            
            # Calculate win probability
            score_diff = row['predicted_home_score'] - row['predicted_away_score']
            # Simple logistic function for win probability
            win_prob = 1 / (1 + np.exp(-0.1 * score_diff))
            live_games_df.at[idx, 'win_probability'] = win_prob
            
            # Calculate confidence based on quarter
            if current_quarter == 0:
                confidence = 0.5  # Pre-game
            elif current_quarter == 1:
                confidence = 0.6  # 1st quarter
            elif current_quarter == 2:
                confidence = 0.7  # 2nd quarter
            elif current_quarter == 3:
                confidence = 0.85  # 3rd quarter
            else:
                confidence = 0.95  # 4th quarter
            
            live_games_df.at[idx, 'prediction_confidence'] = confidence
            
            # If we have actual final scores (for testing), calculate errors
            if 'actual_final_home' in row and 'actual_final_away' in row:
                live_games_df.at[idx, 'home_score_error'] = row['predicted_home_score'] - row['actual_final_home']
                live_games_df.at[idx, 'away_score_error'] = row['predicted_away_score'] - row['actual_final_away']
                
                # Calculate percentage error
                if row['actual_final_home'] > 0:
                    live_games_df.at[idx, 'home_error_pct'] = abs(row['home_score_error'] / row['actual_final_home']) * 100
                if row['actual_final_away'] > 0:
                    live_games_df.at[idx, 'away_error_pct'] = abs(row['away_score_error'] / row['actual_final_away']) * 100

        # 6. Generate recommendations
        for idx, row in live_games_df.iterrows():
            try:
                model_outputs = {
                    "win_probability": row['win_probability'],
                    "momentum_shift": row.get('cumulative_momentum', 0),
                    "projected_margin": row['predicted_home_score'] - row['predicted_away_score'],
                    "total_projected_score": row['predicted_home_score'] + row['predicted_away_score'],
                    "quarter": row.get('current_quarter', 0),
                    "time_remaining": None  # Would need to be provided by data source
                }
                
                recommendations = generate_recommendations(model_outputs)
                
                # Store top recommendations
                for rec_key, rec_value in recommendations.items():
                    live_games_df.at[idx, f'rec_{rec_key}'] = rec_value
            except Exception as e:
                print(f"Error generating recommendations for game {row.get('game_id')}: {e}")

    except Exception as e:
        print(f"Error making predictions: {e}")
        traceback.print_exc()

    # Return the dataframe with predictions
    return live_games_df

In [ ]:
# Cell 16: Enhanced Monitoring with Improved Validation

def monitor_live_games(update_interval=60, max_iterations=10, run_validation=False, show_recommendations=True):
    """
    Enhanced monitor function with improved schedule handling and data validation
    """
    # Initialize prediction history if not already in globals
    if 'prediction_history' not in globals():
        global prediction_history
        prediction_history = {}
    
    # Apply season scoring adjustment
    try:
        season_adjustment = 1.0  # Default to no adjustment
        print(f"Using season adjustment factor: {season_adjustment:.3f}")
    except Exception as e:
        print(f"Error getting season adjustment: {e}")
        season_adjustment = 1.0
    
    # Run validation if requested
    if run_validation:
        print("Running validation on historical data first...")
        try:
            # Find some historical games to validate with
            validation_games = find_recent_games_for_testing(limit=10)
            if not validation_games.empty:
                # Run predictions on historical games
                validation_predictions = run_enhanced_prediction_pipeline()
                
                if not validation_predictions.empty and 'home_score_error' in validation_predictions.columns:
                    mean_error = validation_predictions['home_score_error'].abs().mean()
                    print(f"Validation complete. Mean absolute error: {mean_error:.2f} points")
                    print("Model validation complete. Ready for live prediction.")
                else:
                    print("Validation generated predictions but couldn't calculate error metrics.")
            else:
                print("Validation couldn't find suitable historical games.")
        except Exception as e:
            print(f"Error during validation: {e}")
            traceback.print_exc()
    
    print("Fetching team rolling averages...")
    team_avgs = get_team_rolling_averages()
    print(f"Team rolling averages loaded for {len(team_avgs)} teams")
    
    # Load the official schedule before starting
    try:
        global OFFICIAL_SCHEDULE
        if OFFICIAL_SCHEDULE is None or OFFICIAL_SCHEDULE.empty:
            OFFICIAL_SCHEDULE = load_and_cache_official_schedule()
    except Exception as e:
        print(f"Error loading official schedule: {e}")
        traceback.print_exc()
    
    # Main monitoring loop
    for i in range(max_iterations):
        print(f"\nUpdate #{i+1} - Fetching live game data...")
        
        # Run the full pipeline
        predictions = run_enhanced_prediction_pipeline()
        
        if predictions.empty:
            print("No predictions generated. Waiting for next update...")
            time.sleep(update_interval)
            continue
        
        # Store predictions in history
        for _, game in predictions.iterrows():
            game_key = f"{game['home_team']} vs {game['away_team']}"
            current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            
            if game_key not in prediction_history:
                prediction_history[game_key] = []
            
            # Add current prediction
            prediction_history[game_key].append({
                'timestamp': current_time,
                'current_quarter': game['current_quarter'],
                'current_home_score': game['current_home_score'] if 'current_home_score' in game else game.get('home_score', 0),
                'current_away_score': game['current_away_score'] if 'current_away_score' in game else game.get('away_score', 0),
                'predicted_home_score': game['predicted_home_score'],
                'predicted_away_score': game['predicted_away_score'],
                'win_probability': game.get('win_probability', 0.5)
            })
        
        # Display recommendations if requested
        if show_recommendations:
            recommendations_by_game = {}
            
            for _, game in predictions.iterrows():
                # Get stored recommendations if available
                recs = {}
                for col in game.index:
                    if col.startswith('rec_'):
                        rec_key = col[4:]  # Remove 'rec_' prefix
                        recs[rec_key] = game[col]
                
                # If no recommendations were stored, generate them
                if not recs:
                    try:
                        model_outputs = {
                            "win_probability": float(game.get('win_probability', 0.5)),
                            "momentum_shift": float(game.get('cumulative_momentum', 0)),
                            "projected_margin": float(game.get('predicted_home_score', 0) - game.get('predicted_away_score', 0)),
                            "total_projected_score": float(game.get('predicted_home_score', 0) + game.get('predicted_away_score', 0)),
                            "quarter": int(game.get('current_quarter', 0)),
                            "time_remaining": None
                        }
                        
                        recs = generate_recommendations(model_outputs)
                    except Exception as e:
                        print(f"Error generating recommendations: {e}")
                        # Simple default recommendations
                        score_diff = game.get('predicted_home_score', 0) - game.get('predicted_away_score', 0)
                        win_prob = game.get('win_probability', 0.5)
                        if win_prob > 0.7:
                            betting_tip = "Home team strong favorite."
                        elif win_prob < 0.3:
                            betting_tip = "Home team significant underdog."
                        else:
                            betting_tip = "Game appears competitive."
                            
                        recs = {
                            "betting_tip": betting_tip,
                            "margin": f"Projected margin: {score_diff:.1f} points"
                        }
                
                game_name = f"{game['home_team']} vs {game['away_team']}"
                recommendations_by_game[game_name] = recs
            
            # Display recommendations
            print("\n=== GAME RECOMMENDATIONS ===")
            for game_name, recs in recommendations_by_game.items():
                print(f"\n{game_name}:")
                for rec_type, recommendation in recs.items():
                    if recommendation:  # Only show non-empty recommendations
                        print(f"  • {rec_type}: {recommendation}")
        
        # Visualize predictions 
        try:
            # First, display a simple text summary
            print("\n=== PREDICTION RESULTS ===")
            for _, game in predictions.iterrows():
                current_q = game.get('current_quarter', 0)
                q_str = f"Quarter {current_q}" if current_q > 0 else "Pre-game"
                
                print(f"\n{game['home_team']} vs {game['away_team']} - {q_str}")
                
                if current_q > 0:
                    print(f"Current: {game['home_team']} {game.get('current_home_score', game.get('home_score', 0)):.0f} - " 
                          f"{game['away_team']} {game.get('current_away_score', game.get('away_score', 0)):.0f}")
                
                print(f"Predicted: {game['home_team']} {game['predicted_home_score']:.1f} - " 
                      f"{game['away_team']} {game['predicted_away_score']:.1f}")
                
                if 'win_probability' in game:
                    win_pct = game['win_probability'] * 100
                    print(f"Win Probability: {win_pct:.1f}%")
                
                # If we have error metrics available, show them
                if 'home_score_error' in game:
                    print(f"Validation - Prediction Error: {game['home_score_error']:.1f} points")
            
            # If we have at least one game with multiple predictions, plot trend
            has_history = False
            for game_key, history in prediction_history.items():
                if len(history) > 1:
                    has_history = True
                    break
                    
            if has_history:
                try:
                    plt.figure(figsize=(12, 6))
                    
                    # For each game, plot prediction evolution
                    for game_key, history in prediction_history.items():
                        if len(history) > 1:
                            times = range(len(history))
                            home_preds = [p['predicted_home_score'] for p in history]
                            away_preds = [p['predicted_away_score'] for p in history]
                            
                            plt.plot(times, home_preds, 'b-', label=f"{game_key} - Home")
                            plt.plot(times, away_preds, 'r-', label=f"{game_key} - Away")
                    
                    plt.xlabel('Update Index')
                    plt.ylabel('Predicted Score')
                    plt.title('Prediction Evolution Over Time')
                    plt.legend()
                    plt.grid(True, alpha=0.3)
                    plt.tight_layout()
                    plt.show()
                except Exception as e:
                    print(f"Error plotting prediction trends: {e}")
                    
        except Exception as e:
            print(f"Error visualizing predictions: {e}")
            traceback.print_exc()
        
        # Wait for next update
        if i < max_iterations - 1:  # Don't wait after the last iteration
            print(f"\nWaiting {update_interval} seconds for next update...")
            time.sleep(update_interval)
    
    print("Monitoring complete.")
    return prediction_history

In [ ]:
# Cell 17: Improved Integration Test

def test_integration():
    """
    Tests the integrated prediction and recommendation system with a simulated game
    """
    print("Testing integrated in-game prediction system...")
    
    # Create a simulated game in progress with realistic quarter scores
    simulated_game = {
        'game_id': 9999,
        'home_team': 'Boston Celtics',
        'away_team': 'Los Angeles Lakers',
        'game_date': datetime.now().strftime('%Y-%m-%d'),
        'home_q1': 28,
        'home_q2': 27,
        'home_q3': 22,
        'home_q4': 0,  # 4th quarter not played yet
        'away_q1': 24,
        'away_q2': 29,
        'away_q3': 26,
        'away_q4': 0,  # 4th quarter not played yet
        'current_quarter': 3,  # End of 3rd quarter
        'matched_to_schedule': True  # Pretend it matched schedule
    }
    
    # Calculate current score
    simulated_game['home_score'] = simulated_game['home_q1'] + simulated_game['home_q2'] + simulated_game['home_q3']
    simulated_game['away_score'] = simulated_game['away_q1'] + simulated_game['away_q2'] + simulated_game['away_q3']
    
    # Create DataFrame
    sim_df = pd.DataFrame([simulated_game])
    
    # Calculate score ratio
    score_ratio = simulated_game['home_score'] / (simulated_game['home_score'] + simulated_game['away_score'])
    sim_df['score_ratio'] = score_ratio
    
    # Get team averages
    team_avgs = get_team_rolling_averages()
    
    # Add rolling scores
    sim_df['rolling_home_score'] = team_avgs.get(simulated_game['home_team'], 105.0)
    sim_df['rolling_away_score'] = team_avgs.get(simulated_game['away_team'], 105.0)
    
    # Add rest and momentum features
    try:
        # Rest features
        home_rest = get_rest_data(simulated_game['home_team'], simulated_game['game_date'])
        away_rest = get_rest_data(simulated_game['away_team'], simulated_game['game_date'])
        
        sim_df['rest_days_home'] = home_rest['rest_days']
        sim_df['rest_days_away'] = away_rest['rest_days']
        sim_df['is_back_to_back_home'] = int(home_rest['is_back_to_back'])
        sim_df['is_back_to_back_away'] = int(away_rest['is_back_to_back'])
        sim_df['rest_advantage'] = home_rest['rest_days'] - away_rest['rest_days']
        
        # Previous matchup difference
        sim_df['prev_matchup_diff'] = get_previous_matchup_diff(
            simulated_game['home_team'], simulated_game['away_team'])
        
        # Momentum features
        home_q1 = simulated_game['home_q1']
        home_q2 = simulated_game['home_q2']
        home_q3 = simulated_game['home_q3']
        away_q1 = simulated_game['away_q1']
        away_q2 = simulated_game['away_q2']
        away_q3 = simulated_game['away_q3']
        
        q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
        q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
        
        sim_df['q1_to_q2_momentum'] = q1_to_q2
        sim_df['q2_to_q3_momentum'] = q2_to_q3
        sim_df['q3_to_q4_momentum'] = 0
        
        # Weighted cumulative momentum
        weights = [0.3, 0.7]  # More weight on recent quarters
        cum_momentum = (q1_to_q2 * weights[0] + q2_to_q3 * weights[1]) / sum(weights)
        sim_df['cumulative_momentum'] = max(min(cum_momentum / 15.0, 1.0), -1.0)  # Normalize to [-1, 1]
    except Exception as e:
        print(f"Error adding enhanced features: {e}")
        # Set defaults
        sim_df['rest_days_home'] = 2
        sim_df['rest_days_away'] = 2
        sim_df['is_back_to_back_home'] = 0
        sim_df['is_back_to_back_away'] = 0
        sim_df['rest_advantage'] = 0
        sim_df['prev_matchup_diff'] = 0
        sim_df['q1_to_q2_momentum'] = 0
        sim_df['q2_to_q3_momentum'] = 0
        sim_df['q3_to_q4_momentum'] = 0
        sim_df['cumulative_momentum'] = 0
    
    # Load model
    try:
        model = None
        if 'model' in globals() and globals()['model'] is not None:
            model = globals()['model']
        else:
            # Try to load from config path
            try:
                model_path = config.MODEL_PATH
                if os.path.exists(model_path):
                    model = joblib.load(model_path)
                    print(f"Loaded model from {model_path}")
            except Exception as model_load_err:
                print(f"Error loading model from config path: {model_load_err}")
                
                # Try common locations
                common_paths = [
                    './model.pkl',
                    './final_xgb_model.pkl',
                    './models/score_prediction_model.pkl',
                    './enhanced_xgb_model.pkl'
                ]
                
                for path in common_paths:
                    if os.path.exists(path):
                        try:
                            model = joblib.load(path)
                            print(f"Loaded model from {path}")
                            break
                        except:
                            pass
                
        if model is None:
            print("Could not load model for prediction. Using mock prediction.")
            # Mock predictions based on current score
            sim_df['predicted_home_score'] = simulated_game['home_score'] * 1.25  # Estimated final 25% more
            sim_df['predicted_away_score'] = simulated_game['away_score'] * 1.25
        else:
            # Determine which features to use
            is_enhanced_model = False
            if hasattr(model, 'feature_importances_'):
                feature_count = len(model.feature_importances_)
                is_enhanced_model = (feature_count > 8)
            
            if is_enhanced_model:
                features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
            else:
                features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
            
            # Make sure all features exist
            for feature in features:
                if feature not in sim_df.columns:
                    sim_df[feature] = 0
            
            # Make predictions
            X = sim_df[features]
            home_preds = model.predict(X)
            sim_df['predicted_home_score'] = home_preds
            
            # Estimate away score
            home_court_advantage = 3.5
            matchup_diff = sim_df['prev_matchup_diff'].values[0]
            sim_df['predicted_away_score'] = home_preds - home_court_advantage - matchup_diff
    
    except Exception as e:
        print(f"Error during prediction: {e}")
        traceback.print_exc()
        return None
    
    # Calculate win probability
    score_diff = sim_df['predicted_home_score'].values[0] - sim_df['predicted_away_score'].values[0]
    win_prob = 1 / (1 + np.exp(-0.1 * score_diff))
    sim_df['win_probability'] = win_prob
    
    # Generate recommendations
    try:
        model_outputs = {
            "win_probability": win_prob,
            "momentum_shift": sim_df['cumulative_momentum'].values[0],
            "projected_margin": score_diff,
            "total_projected_score": sim_df['predicted_home_score'].values[0] + sim_df['predicted_away_score'].values[0],
            "quarter": 3,  # End of 3rd quarter
            "time_remaining": 12  # One quarter (12 minutes) remaining
        }
        
        recommendations = generate_recommendations(model_outputs)
    except Exception as e:
        print(f"Error generating recommendations: {e}")
        recommendations = {
            "betting_tip": "Game appears competitive.",
            "momentum_advice": f"Current momentum: {sim_df['cumulative_momentum'].values[0]:.2f}",
            "spread_tip": f"Projected margin: {score_diff:.1f} points"
        }
    
    # Display complete results
    print("\n=== PREDICTION RESULTS ===")
    print(f"Game: {simulated_game['home_team']} vs {simulated_game['away_team']}")
    print(f"Current Score: {simulated_game['home_team']} {simulated_game['home_score']} - {simulated_game['away_team']} {simulated_game['away_score']}")
    print(f"Current Quarter: {simulated_game['current_quarter']}")
    print(f"Predicted Final: {simulated_game['home_team']} {sim_df['predicted_home_score'].values[0]:.1f} - {simulated_game['away_team']} {sim_df['predicted_away_score'].values[0]:.1f}")
    print(f"Win Probability: {win_prob:.1%}")
    print(f"Momentum: {sim_df['cumulative_momentum'].values[0]:.2f}")
    
    print("\n=== RECOMMENDATIONS ===")
    for rec_type, recommendation in recommendations.items():
        print(f"  • {rec_type}: {recommendation}")
    
    return sim_df, recommendations

# Run the improved test
try:
    test_results = test_integration()
except Exception as e:
    print(f"Integration test failed: {e}")
    traceback.print_exc()

In [ ]:
# Cell 18 - Train Enhanced Model with New Features

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import joblib
from sqlalchemy import create_engine, text
import config

print("Training enhanced in-game prediction model with new features...")

# Connect to the database
engine = create_engine(config.DATABASE_URL)

# Safely load the historical game data with fixed SQL query
# The previous error was due to a PostgreSQL function compatibility issue
try:
    # Use simpler query without DATE_PART function
    query = """
    SELECT g.* 
    FROM nba_historical_game_stats g
    ORDER BY g.game_date
    """
    
    df = pd.read_sql(text(query), engine)
    print(f"Successfully loaded {len(df)} historical games")
    
    # Add rest features manually using pandas operations
    print("Adding rest-related features...")
    df['game_date'] = pd.to_datetime(df['game_date'])
    df = df.sort_values(['game_date'])
    
    # Initialize rest features
    df['rest_days_home'] = 2  # Default: 2 days rest
    df['rest_days_away'] = 2  # Default: 2 days rest 
    df['is_home_b2b'] = 0
    df['is_away_b2b'] = 0
    
    # Get unique teams
    teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
    
    # Calculate rest days for each team
    for team in teams:
        # Get games where team is at home
        home_games = df[df['home_team'] == team][['game_id', 'game_date']].copy()
        home_games['team_role'] = 'home'
        
        # Get games where team is away
        away_games = df[df['away_team'] == team][['game_id', 'game_date']].copy()
        away_games['team_role'] = 'away'
        
        # Combine and sort chronologically
        all_games = pd.concat([home_games, away_games]).sort_values('game_date')
        
        # Calculate days between games
        all_games['prev_date'] = all_games['game_date'].shift(1)
        all_games['days_rest'] = (all_games['game_date'] - all_games['prev_date']).dt.days
        
        # Fill NaN values for first game
        all_games['days_rest'] = all_games['days_rest'].fillna(5)
        
        # Cap rest days to reasonable values
        all_games['days_rest'] = all_games['days_rest'].clip(1, 10)
        
        # Flag back-to-back games
        all_games['is_b2b'] = (all_games['days_rest'] == 1).astype(int)
        
        # Update main dataframe
        for idx, row in all_games.iterrows():
            game_id = row['game_id']
            days_rest = row['days_rest']
            is_b2b = row['is_b2b']
            
            if row['team_role'] == 'home':
                mask = (df['game_id'] == game_id) & (df['home_team'] == team)
                df.loc[mask, 'rest_days_home'] = days_rest
                df.loc[mask, 'is_home_b2b'] = is_b2b
            else:
                mask = (df['game_id'] == game_id) & (df['away_team'] == team)
                df.loc[mask, 'rest_days_away'] = days_rest
                df.loc[mask, 'is_away_b2b'] = is_b2b
    
    # Calculate rest advantage
    df['rest_advantage'] = df['rest_days_home'] - df['rest_days_away']
    
    # Cap extreme values
    df['rest_advantage'] = df['rest_advantage'].clip(-5, 5)
    
    print("Rest features calculation complete.")

    # Calculate momentum features
    print("Adding momentum-related features...")
    
    # Calculate quarter-to-quarter momentum shifts
    df['q1_to_q2_momentum'] = (df['home_q2'] - df['home_q1']) - (df['away_q2'] - df['away_q1'])
    df['q2_to_q3_momentum'] = (df['home_q3'] - df['home_q2']) - (df['away_q3'] - df['away_q2'])
    df['q3_to_q4_momentum'] = (df['home_q4'] - df['home_q3']) - (df['away_q4'] - df['away_q3'])
    
    # Cap extreme values
    df['q1_to_q2_momentum'] = df['q1_to_q2_momentum'].clip(-20, 20)
    df['q2_to_q3_momentum'] = df['q2_to_q3_momentum'].clip(-20, 20)
    df['q3_to_q4_momentum'] = df['q3_to_q4_momentum'].clip(-20, 20)
    
    # Calculate cumulative momentum (weighted)
    df['cumulative_momentum'] = (
        df['q1_to_q2_momentum'] * 0.2 + 
        df['q2_to_q3_momentum'] * 0.3 + 
        df['q3_to_q4_momentum'] * 0.5
    )
    
    # Normalize to [-1, 1]
    df['cumulative_momentum'] = df['cumulative_momentum'] / 15.0
    df['cumulative_momentum'] = df['cumulative_momentum'].clip(-1, 1)
    
    print("Momentum features calculation complete.")
    
    # Calculate matchup differences
    print("Calculating matchup differences...")
    
    df['prev_matchup_diff'] = 0.0
    
    # Process unique team pairs
    processed_pairs = set()
    
    for idx, row in df.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        team_pair = tuple(sorted([home_team, away_team]))
        
        # Skip if already processed
        if team_pair in processed_pairs:
            continue
            
        # Find all games between these teams
        team_games = df[
            ((df['home_team'] == home_team) & (df['away_team'] == away_team)) |
            ((df['home_team'] == away_team) & (df['away_team'] == home_team))
        ].sort_values('game_date')
        
        # Update each game with previous matchup data
        for g_idx, game in team_games.iterrows():
            # Find games prior to this one
            prev_games = team_games[team_games['game_date'] < game['game_date']]
            
            if len(prev_games) > 0:
                # Calculate point differentials from home team perspective
                diffs = []
                for _, prev in prev_games.iterrows():
                    if prev['home_team'] == game['home_team']:
                        # Same home team
                        diff = prev['home_score'] - prev['away_score']
                    else:
                        # Teams reversed
                        diff = prev['away_score'] - prev['home_score']
                    diffs.append(diff)
                
                # Calculate mean and cap extreme values
                avg_diff = np.mean(diffs)
                avg_diff = np.clip(avg_diff, -15, 15)
                
                # Update the dataframe
                df.loc[g_idx, 'prev_matchup_diff'] = avg_diff
        
        processed_pairs.add(team_pair)
    
    print(f"Matchup differences calculated for {len(processed_pairs)} team pairs.")
    
    # Calculate score ratio
    df['score_ratio'] = df['home_score'] / (df['home_score'] + df['away_score'])
    
    # Define features and target
    print("Preparing data for model training...")
    
    features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4',
        'score_ratio', 'prev_matchup_diff',
        'rest_days_home', 'rest_days_away', 'rest_advantage',
        'is_home_b2b', 'is_away_b2b',
        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
    ]
    
    target = 'home_score'
    
    # Make sure all feature columns exist and are numeric
    for col in features:
        if col not in df.columns:
            print(f"Missing feature column: {col}")
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Prepare data
    X = df[features]
    y = df[target]
    
    # Split data (time-based)
    train_size = int(0.8 * len(df))
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
    
    print(f"Training set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")
    
    # Train model
    model = GradientBoostingRegressor(
        n_estimators=200, 
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        subsample=0.8
    )
    
    model.fit(X_train, y_train)
    
    # Evaluate model
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, train_preds)
    test_mse = mean_squared_error(y_test, test_preds)
    r2 = r2_score(y_test, test_preds)
    
    print(f"Training MSE: {train_mse:.2f}")
    print(f"Test MSE: {test_mse:.2f}")
    print(f"R² Score: {r2:.4f}")
    
    # Save the enhanced model
    enhanced_model_path = 'enhanced_xgb_model.pkl'
    joblib.dump(model, enhanced_model_path)
    print(f"Enhanced model saved to {enhanced_model_path}")
    
    # Make the model available globally
    globals()['model'] = model
    
except Exception as e:
    print(f"Error in model training: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 19 - Visualize Feature Importance and Test Quarter-Specific Performance

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# Visualize feature importance
if 'model' in globals() and hasattr(model, 'feature_importances_'):
    # Get feature importances
    feature_importances = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importances)
    plt.title('Feature Importance in Enhanced In-Game Model')
    plt.tight_layout()
    plt.show()
    
    # Group features by type
    feature_groups = {
        'Quarter Scores': ['home_q1', 'home_q2', 'home_q3', 'home_q4'],
        'Game State': ['score_ratio'],
        'Matchup History': ['prev_matchup_diff'],
        'Rest': ['home_rest_days', 'away_rest_days', 'rest_advantage', 'is_home_b2b', 'is_away_b2b'],
        'Momentum': ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']
    }
    
    group_importance = {}
    for group, feats in feature_groups.items():
        # Sum importance of features in this group
        group_importance[group] = sum(
            model.feature_importances_[features.index(f)] 
            for f in feats if f in features
        )
    
    # Create pie chart of feature group importance
    plt.figure(figsize=(10, 8))
    plt.pie(
        group_importance.values(), 
        labels=group_importance.keys(), 
        autopct='%1.1f%%', 
        shadow=True, 
        startangle=140
    )
    plt.axis('equal')
    plt.title('Feature Group Contribution to In-Game Predictions')
    plt.tight_layout()
    plt.show()

# Analyze model performance by quarter
print("\nAnalyzing performance by quarter...")

# Create predictions for different quarter scenarios
quarter_analysis = []

for current_quarter in range(1, 5):
    # Filter test data to include only information available up to current_quarter
    quarter_X_test = X_test.copy()
    
    # Zero out information from future quarters
    for q in range(current_quarter + 1, 5):
        quarter_col = f'home_q{q}'
        if quarter_col in quarter_X_test.columns:
            quarter_X_test[quarter_col] = 0
    
    # Zero out momentum features that wouldn't be available
    if current_quarter < 2:
        quarter_X_test['q1_to_q2_momentum'] = 0
        quarter_X_test['q2_to_q3_momentum'] = 0
        quarter_X_test['q3_to_q4_momentum'] = 0
        quarter_X_test['cumulative_momentum'] = 0
    elif current_quarter < 3:
        quarter_X_test['q2_to_q3_momentum'] = 0
        quarter_X_test['q3_to_q4_momentum'] = 0
        # Keep q1_to_q2_momentum and recalculate cumulative_momentum
        quarter_X_test['cumulative_momentum'] = quarter_X_test['q1_to_q2_momentum']
    elif current_quarter < 4:
        quarter_X_test['q3_to_q4_momentum'] = 0
        # Recalculate cumulative_momentum with just q1->q2 and q2->q3
        quarter_X_test['cumulative_momentum'] = (
            quarter_X_test['q1_to_q2_momentum'] * 0.2 + 
            quarter_X_test['q2_to_q3_momentum'] * 0.3
        ) / 0.5
    
    # Generate predictions
    quarter_preds = model.predict(quarter_X_test)
    
    # Calculate metrics
    quarter_mse = mean_squared_error(y_test, quarter_preds)
    quarter_mae = np.mean(np.abs(y_test - quarter_preds))
    
    quarter_analysis.append({
        'Quarter': current_quarter,
        'MSE': quarter_mse,
        'MAE': quarter_mae,
        'RMSE': np.sqrt(quarter_mse)
    })

# Display quarter-by-quarter performance
quarter_df = pd.DataFrame(quarter_analysis)
print(quarter_df)

# Plot error by quarter
plt.figure(figsize=(10, 6))
plt.bar(quarter_df['Quarter'], quarter_df['RMSE'], color='salmon')
plt.xlabel('Information Available Through Quarter')
plt.ylabel('Root Mean Square Error')
plt.title('Prediction Error by Available Quarter Information')
plt.xticks([1, 2, 3, 4])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 20 - Fetch Schedule Data

def fetch_scheduled_games(date_str=None):
    """Fetch scheduled games from the database for a specific date (in Pacific Time)."""
    import pytz
    
    # If no date provided, use today in Pacific Time
    if date_str is None:
        pacific_tz = pytz.timezone('America/Los_Angeles')
        date_str = datetime.now(pacific_tz).strftime('%Y-%m-%d')
    
    print(f"Fetching scheduled games for {date_str} (Pacific Time)")
    
    try:
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute()
        if response.data:
            return pd.DataFrame(response.data)
        else:
            print(f"No scheduled games found for {date_str}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error fetching scheduled games: {e}")
        return pd.DataFrame()

# Test fetching today's schedule
today_schedule = fetch_scheduled_games()
display(today_schedule)

In [ ]:
# Cell 21 (Alternate): Enhanced Feature Engineering - Compute Momentum Features

def compute_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute momentum-based features for prediction."""
    features_df = df.copy()
    features_df['q1_to_q2_momentum'] = 0
    features_df['q2_to_q3_momentum'] = 0
    features_df['q3_to_q4_momentum'] = 0
    features_df['cumulative_momentum'] = 0
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        h = [float(row.get(f'home_q{i}', 0)) for i in range(1, 5)]
        a = [float(row.get(f'away_q{i}', 0)) for i in range(1, 5)]
        if current_quarter >= 2:
            features_df.at[idx, 'q1_to_q2_momentum'] = (h[1] - h[0]) - (a[1] - a[0])
        if current_quarter >= 3:
            features_df.at[idx, 'q2_to_q3_momentum'] = (h[2] - h[1]) - (a[2] - a[1])
        if current_quarter >= 4:
            features_df.at[idx, 'q3_to_q4_momentum'] = (h[3] - h[2]) - (a[3] - a[2])
        weights = [0.2, 0.3, 0.5]
        if current_quarter == 2:
            cum = features_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            cum = (features_df.at[idx, 'q1_to_q2_momentum']*weights[0] + features_df.at[idx, 'q2_to_q3_momentum']*weights[1]) / (weights[0]+weights[1])
        elif current_quarter >= 4:
            cum = (features_df.at[idx, 'q1_to_q2_momentum']*weights[0] + features_df.at[idx, 'q2_to_q3_momentum']*weights[1] + features_df.at[idx, 'q3_to_q4_momentum']*weights[2]) / sum(weights)
        else:
            cum = 0
        features_df.at[idx, 'cumulative_momentum'] = np.clip(cum/15.0, -1, 1)
    return features_df


In [ ]:
# Cell 21: Enhanced feature engineering
def compute_momentum_features(df):
    """Compute momentum-based features for prediction."""
    features_df = df.copy()
    
    # Add momentum columns
    features_df['q1_to_q2_momentum'] = 0
    features_df['q2_to_q3_momentum'] = 0
    features_df['q3_to_q4_momentum'] = 0
    features_df['cumulative_momentum'] = 0
    
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Quarter scores
        home_q1 = float(row.get('home_q1', 0))
        home_q2 = float(row.get('home_q2', 0))
        home_q3 = float(row.get('home_q3', 0))
        home_q4 = float(row.get('home_q4', 0))
        
        away_q1 = float(row.get('away_q1', 0))
        away_q2 = float(row.get('away_q2', 0))
        away_q3 = float(row.get('away_q3', 0))
        away_q4 = float(row.get('away_q4', 0))
        
        # Calculate quarter-to-quarter momentum
        if current_quarter >= 2:
            features_df.at[idx, 'q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
            
        if current_quarter >= 3:
            features_df.at[idx, 'q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
            
        if current_quarter >= 4:
            features_df.at[idx, 'q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
        
        # Weighted momentum calculation
        weights = [0.2, 0.3, 0.5]  # Weight recent quarters more heavily
        
        if current_quarter == 2:
            features_df.at[idx, 'cumulative_momentum'] = features_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            features_df.at[idx, 'cumulative_momentum'] = (
                features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
            ) / (weights[0] + weights[1])
        elif current_quarter >= 4:
            features_df.at[idx, 'cumulative_momentum'] = (
                features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
            ) / sum(weights)
        
        # Normalize momentum to range [-1, 1]
        if features_df.at[idx, 'cumulative_momentum'] != 0:
            max_momentum = 15.0  # Typical max quarter differential
            features_df.at[idx, 'cumulative_momentum'] = max(min(
                features_df.at[idx, 'cumulative_momentum'] / max_momentum, 1.0
            ), -1.0)
    
    return features_df

In [ ]:
# Cell 22 - Define fetch_recent_games_for_testing

def fetch_recent_games_for_testing(limit=5):
    """Find recent completed games to use for testing the prediction model."""
    try:
        # Look back up to 5 days for recent games
        dates_to_try = []
        today = datetime.now()
        
        for i in range(1, 6):
            date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
            dates_to_try.append(date)
        
        # Try each date until we find games
        for test_date in dates_to_try:
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .eq("game_date", test_date)\
                .limit(limit).execute()
            
            if response.data:
                print(f"Found {len(response.data)} historical games from {test_date}")
                
                # Convert to DataFrame
                historical_df = pd.DataFrame(response.data)
                
                # Simulate as in-progress games
                import random
                live_games = []
                
                for _, game in historical_df.iterrows():
                    # Randomly select a quarter (1-4) for simulation
                    sim_quarter = random.randint(1, 4)
                    
                    # Create simulated live game with data up to selected quarter
                    sim_game = {
                        'game_id': game['game_id'],
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': sim_quarter,
                        'simulated': True  # Flag as simulated
                    }
                    
                    # Add quarter scores up to simulated quarter
                    for q in range(1, 5):
                        if q <= sim_quarter:
                            sim_game[f'home_q{q}'] = game.get(f'home_q{q}', 0)
                            sim_game[f'away_q{q}'] = game.get(f'away_q{q}', 0)
                        else:
                            sim_game[f'home_q{q}'] = 0
                            sim_game[f'away_q{q}'] = 0
                    
                    # Calculate current score based on visible quarters
                    sim_game['home_score'] = sum(
                        [sim_game.get(f'home_q{q}', 0) for q in range(1, sim_quarter+1)]
                    )
                    sim_game['away_score'] = sum(
                        [sim_game.get(f'away_q{q}', 0) for q in range(1, sim_quarter+1)]
                    )
                    
                    # Store actual final scores for validation
                    sim_game['actual_home_final'] = game['home_score']
                    sim_game['actual_away_final'] = game['away_score']
                    
                    live_games.append(sim_game)
                
                simulated_df = pd.DataFrame(live_games)
                simulated_df['verified'] = True  # Mark as verified since these are from historical data
                return simulated_df
                
        print("No recent games found for testing")
        return pd.DataFrame()
        
    except Exception as e:
        print(f"Error finding historical games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

In [ ]:
# Cell 23 - Define load_model function

def load_model():
    """Load the trained score prediction model."""
    try:
        import joblib
        import os
        
        model_path = config.MODEL_PATH
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            print(f"Model loaded from {model_path}")
            return model
        else:
            # Try to load a model from a different location
            model_path = os.path.join(os.getcwd(), 'models', 'score_prediction_model.pkl')
            if os.path.exists(model_path):
                model = joblib.load(model_path)
                print(f"Model loaded from {model_path}")
                return model
            
            print(f"Model file not found at {model_path}")
            return None
    except Exception as e:
        print(f"Error loading model: {e}")
        traceback.print_exc()
        return None

In [ ]:
# Cell 24 - Define run_predictions and supporting functions

def get_team_rolling_averages(days_lookback=60):
    """Get rolling scoring averages for all teams."""
    try:
        # Get recent games from the last 60 days
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        
        if not response.data:
            print("No historical data found for team averages")
            return {}
        
        # Calculate team averages
        team_avgs = {}
        df = pd.DataFrame(response.data)
        
        # Get unique teams
        home_teams = set(df['home_team'].unique())
        away_teams = set(df['away_team'].unique())
        all_teams = home_teams.union(away_teams)
        
        for team in all_teams:
            # Get home and away games
            home_games = df[df['home_team'] == team]['home_score']
            away_games = df[df['away_team'] == team]['away_score']
            
            # Combine and calculate average
            all_scores = pd.concat([home_games, away_games])
            if len(all_scores) > 0:
                team_avgs[team] = all_scores.mean()
            else:
                team_avgs[team] = 105.0  # Default NBA average
        
        return team_avgs
    
    except Exception as e:
        print(f"Error getting team averages: {e}")
        traceback.print_exc()
        return {}

def get_previous_matchup_diff(home_team, away_team, max_lookback=5):
    """Gets the point differential from previous matchups between two teams."""
    try:
        # Use separate queries for home and away configurations to avoid syntax issues
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        
        # Combine results
        home_matchups = response_home.data
        away_matchups = response_away.data
        matchups = home_matchups + away_matchups
        
        # Sort by date (most recent first)
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
        
        if not matchups:
            return 0
        
        # Calculate point differential from home team perspective
        differentials = []
        for game in matchups:
            if game['home_team'] == home_team and game['away_team'] == away_team:
                diff = game['home_score'] - game['away_score']
            elif game['home_team'] == away_team and game['away_team'] == home_team:
                diff = game['away_score'] - game['home_score']
            else:
                continue
            differentials.append(diff)
        
        return sum(differentials) / len(differentials) if differentials else 0
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0

def prepare_features(games_df, model=None):
    """Prepare features for prediction based on model type."""
    if games_df.empty:
        return pd.DataFrame()
    
    # Determine if we have the enhanced model
    is_enhanced_model = False
    if model is not None and hasattr(model, 'feature_importances_'):
        feature_count = len(model.feature_importances_)
        is_enhanced_model = (feature_count > 8)
    
    # Define feature lists
    original_features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
    ]
    
    enhanced_features = [
        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
        'score_ratio', 'prev_matchup_diff',
        'rest_days_home', 'rest_days_away', 'rest_advantage',
        'is_back_to_back_home', 'is_back_to_back_away',
        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
    ]
    
    # Choose which features to use
    expected_features = enhanced_features if is_enhanced_model else original_features
    print(f"Using {'enhanced' if is_enhanced_model else 'original'} feature set")
    
    # Get team averages
    team_avgs = get_team_rolling_averages()
    
    # Prepare feature DataFrame
    features = []
    
    for idx, game in games_df.iterrows():
        # Get basic game data
        game_id = game['game_id']
        home_team = game['home_team']
        away_team = game['away_team']
        current_quarter = int(game['current_quarter'])
        
        # Quarter scores
        home_q1 = float(game['home_q1'])
        home_q2 = float(game['home_q2'])
        home_q3 = float(game['home_q3'])
        home_q4 = float(game['home_q4'])
        
        away_q1 = float(game['away_q1'])
        away_q2 = float(game['away_q2'])
        away_q3 = float(game['away_q3'])
        away_q4 = float(game['away_q4'])
        
        # Get matchup history
        prev_matchup_diff = get_previous_matchup_diff(home_team, away_team)
        
        # Calculate score ratio
        score_ratio = game.get('score_ratio', 0.5)
        
        # Create feature dictionary
        feature_dict = {
            'game_id': game_id,
            'home_team': home_team,
            'away_team': away_team,
            'home_q1': home_q1,
            'home_q2': home_q2,
            'home_q3': home_q3,
            'home_q4': home_q4,
            'score_ratio': score_ratio,
            'prev_matchup_diff': prev_matchup_diff
        }
        
        # Add rolling averages for original model
        if not is_enhanced_model:
            feature_dict['rolling_home_score'] = team_avgs.get(home_team, 105.0)
            feature_dict['rolling_away_score'] = team_avgs.get(away_team, 105.0)
        
        # Add enhanced features if needed
        if is_enhanced_model:
            # Default rest features
            feature_dict['rest_days_home'] = 2
            feature_dict['rest_days_away'] = 2
            feature_dict['rest_advantage'] = 0
            feature_dict['is_back_to_back_home'] = 0
            feature_dict['is_back_to_back_away'] = 0
            
            # Calculate momentum features
            feature_dict['q1_to_q2_momentum'] = 0
            feature_dict['q2_to_q3_momentum'] = 0
            feature_dict['q3_to_q4_momentum'] = 0
            feature_dict['cumulative_momentum'] = 0
            
            if current_quarter >= 2:
                feature_dict['q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
                
            if current_quarter >= 3:
                feature_dict['q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
                
            if current_quarter >= 4:
                feature_dict['q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
            
            # Calculate cumulative momentum
            weights = [0.2, 0.3, 0.5]  # Weights for each quarter transition
            
            if current_quarter == 2:
                feature_dict['cumulative_momentum'] = feature_dict['q1_to_q2_momentum']
            elif current_quarter == 3:
                feature_dict['cumulative_momentum'] = (
                    feature_dict['q1_to_q2_momentum'] * weights[0] + 
                    feature_dict['q2_to_q3_momentum'] * weights[1]
                ) / (weights[0] + weights[1])
            elif current_quarter >= 4:
                feature_dict['cumulative_momentum'] = (
                    feature_dict['q1_to_q2_momentum'] * weights[0] + 
                    feature_dict['q2_to_q3_momentum'] * weights[1] + 
                    feature_dict['q3_to_q4_momentum'] * weights[2]
                ) / sum(weights)
            
            # Normalize momentum to [-1, 1]
            if feature_dict['cumulative_momentum'] != 0:
                feature_dict['cumulative_momentum'] = max(min(
                    feature_dict['cumulative_momentum'] / 15.0, 1.0), -1.0)
        
        features.append(feature_dict)
    
    # Create DataFrame
    features_df = pd.DataFrame(features)
    
    # Ensure all expected features exist with proper types
    for feature in expected_features:
        if feature not in features_df.columns:
            features_df[feature] = 0
        features_df[feature] = pd.to_numeric(features_df[feature], errors='coerce').fillna(0)
    
    return features_df

def run_predictions(model, games_df):
    """Run predictions using the model and games data."""
    if games_df.empty:
        print("No games available for prediction")
        return pd.DataFrame()
    
    print(f"Running predictions for {len(games_df)} games...")
    
    # Step 1: Prepare features
    features_df = prepare_features(games_df, model)
    if features_df.empty:
        print("Failed to prepare features")
        return pd.DataFrame()
    
    # Step 2: Get the right feature columns for prediction
    if hasattr(model, 'feature_importances_'):
        feature_count = len(model.feature_importances_)
        is_enhanced = (feature_count > 8)
        
        if is_enhanced:
            model_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
        else:
            model_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
    else:
        # Default to original features
        model_features = [
            'home_q1', 'home_q2', 'home_q3', 'home_q4', 
            'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
        ]
    
    # Make sure all required features exist
    for feature in model_features:
        if feature not in features_df.columns:
            features_df[feature] = 0
    
    # Step 3: Make predictions
    X_pred = features_df[model_features]
    predictions = model.predict(X_pred)
    
    # Step 4: Create results DataFrame with predictions
    results = []
    
    for i, (idx, game) in enumerate(games_df.iterrows()):
        # Get game info
        game_id = game['game_id']
        home_team = game['home_team']
        away_team = game['away_team']
        current_quarter = int(game['current_quarter'])
        home_score = float(game['home_score'])
        away_score = float(game['away_score'])
        
        # Get prediction
        predicted_home_score = predictions[i]
        
        # Calculate away score prediction
        prev_matchup_diff = features_df.loc[i, 'prev_matchup_diff'] if 'prev_matchup_diff' in features_df.columns else 0
        
        # Scale effect based on game progress
        diff_weight = min(0.3 + (0.1 * current_quarter), 0.6)
        
        # Factor in momentum if available
        momentum_adj = 0
        if 'cumulative_momentum' in features_df.columns:
            momentum = features_df.loc[i, 'cumulative_momentum']
            momentum_adj = momentum * 3.0  # Scale to points impact
        
        predicted_away_score = predicted_home_score - (prev_matchup_diff * diff_weight) - momentum_adj
        
        # Ensure predictions are at least current scores
        predicted_home_score = max(predicted_home_score, home_score)
        predicted_away_score = max(predicted_away_score, away_score)
        
        # Calculate win probability
        score_diff = predicted_home_score - predicted_away_score
        game_progress = min(current_quarter / 4.0, 1.0)
        k_factor = 0.05 + (game_progress * 0.15)
        win_probability = 1.0 / (1.0 + np.exp(-k_factor * score_diff))
        
        # Create result dictionary
        result = {
            'game_id': game_id,
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'current_home_score': home_score,
            'current_away_score': away_score,
            'predicted_home_final': predicted_home_score,
            'predicted_away_final': predicted_away_score,
            'remaining_home_points': predicted_home_score - home_score,
            'remaining_away_points': predicted_away_score - away_score,
            'win_probability': win_probability,
            'projected_margin': predicted_home_score - predicted_away_score,
            'total_projected_score': predicted_home_score + predicted_away_score
        }
        
        # Add actual finals if available (for historical testing)
        if 'actual_home_final' in game:
            result['actual_home_final'] = game['actual_home_final']
            result['actual_away_final'] = game['actual_away_final']
        
        results.append(result)
    
    return pd.DataFrame(results)

In [ ]:
# Cell 25: Add Missing Match Functions

def fetch_live_games() -> pd.DataFrame:
    """Fetch live game data from Supabase."""
    try:
        print("Fetching live games from nba_live_game_stats...")
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if response.data:
            print(f"Found {len(response.data)} live games")
            return pd.DataFrame(response.data)
        else:
            print("No live games found.")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def match_live_to_scheduled(live_games_df: pd.DataFrame, scheduled_games_df: pd.DataFrame) -> pd.DataFrame:
    """Match live games to scheduled games using team names."""
    if live_games_df.empty or scheduled_games_df.empty:
        return live_games_df
    matched_games = live_games_df.copy()
    matched_games['verified'] = False
    for idx, game in matched_games.iterrows():
        home_team = str(game.get('home_team', '')).lower()
        away_team = str(game.get('away_team', '')).lower()
        for _, sched in scheduled_games_df.iterrows():
            sched_home = str(sched.get('home_team', '')).lower()
            sched_away = str(sched.get('away_team', '')).lower()
            if (home_team in sched_home and away_team in sched_away) or (home_team in sched_away and away_team in sched_home):
                matched_games.at[idx, 'game_id'] = sched['game_id']
                matched_games.at[idx, 'verified'] = True
                break
    return matched_games

def compute_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute momentum-based features for prediction."""
    features_df = df.copy()
    features_df['q1_to_q2_momentum'] = 0
    features_df['q2_to_q3_momentum'] = 0
    features_df['q3_to_q4_momentum'] = 0
    features_df['cumulative_momentum'] = 0
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        h = [float(row.get(f'home_q{i}', 0)) for i in range(1, 5)]
        a = [float(row.get(f'away_q{i}', 0)) for i in range(1, 5)]
        if current_quarter >= 2:
            features_df.at[idx, 'q1_to_q2_momentum'] = (h[1] - h[0]) - (a[1] - a[0])
        if current_quarter >= 3:
            features_df.at[idx, 'q2_to_q3_momentum'] = (h[2] - h[1]) - (a[2] - a[1])
        if current_quarter >= 4:
            features_df.at[idx, 'q3_to_q4_momentum'] = (h[3] - h[2]) - (a[3] - a[2])
        weights = [0.2, 0.3, 0.5]
        if current_quarter == 2:
            cum = features_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            cum = (features_df.at[idx, 'q1_to_q2_momentum']*weights[0] + features_df.at[idx, 'q2_to_q3_momentum']*weights[1]) / (weights[0]+weights[1])
        elif current_quarter >= 4:
            cum = (features_df.at[idx, 'q1_to_q2_momentum']*weights[0] + features_df.at[idx, 'q2_to_q3_momentum']*weights[1] + features_df.at[idx, 'q3_to_q4_momentum']*weights[2]) / sum(weights)
        else:
            cum = 0
        features_df.at[idx, 'cumulative_momentum'] = np.clip(cum/15.0, -1, 1)
    return features_df


In [ ]:
# Cell 26: Run Updated Pipeline + Analyze Results

def run_enhanced_prediction_pipeline() -> pd.DataFrame:
    """
    Run the full prediction pipeline with all improvements:
      1. Fetch scheduled games.
      2. Fetch live games (or fallback to historical for testing).
      3. Match and verify game data.
      4. Compute enhanced features.
      5. Run predictions.
      6. Return results (with error metrics if available).
    """
    schedule_df = fetch_scheduled_games()
    live_games_df = fetch_live_games()
    if live_games_df.empty:
        print("No live games found. Using historical games for testing...")
        live_games_df = fetch_recent_games_for_testing()
        if live_games_df.empty:
            print("No games available for prediction")
            return pd.DataFrame()
    verified_games = match_live_to_scheduled(live_games_df, schedule_df)
    enhanced_games = compute_momentum_features(verified_games)
    model_loaded = load_model()
    if model_loaded is None:
        print("Failed to load prediction model")
        return pd.DataFrame()
    predictions = run_predictions(model_loaded, enhanced_games)
    if 'actual_home_final' in enhanced_games.columns:
        predictions['home_error'] = predictions['predicted_home_score'] - predictions['actual_home_final']
        predictions['away_error'] = predictions['predicted_away_score'] - predictions['actual_away_final']
        predictions['absolute_error'] = np.abs(predictions['home_error'])
        print(f"Average prediction error: {predictions['absolute_error'].mean():.2f} points")
        for q in range(1, 5):
            q_games = predictions[predictions['current_quarter'] == q]
            if not q_games.empty:
                print(f"Quarter {q} average error: {q_games['absolute_error'].mean():.2f} points")
    return predictions

# Run the pipeline
prediction_results = run_enhanced_prediction_pipeline()
display(prediction_results)
